<style>
.custom-spacing {
    line-height: 0.5;
}
</style>

### <center>**KECERDASAN ARTIFISIAL UNTUK GEOFISIKA**</center>
#### <center>**TUGAS INDIVIDU**</center>

<div style="text-align:center;">
  <img src="https://www.unhas.ac.id/wp-content/uploads/2022/09/Logo-Resmi-Unhas-1.png" alt="Deskripsi Gambar" style="width:280px;">
</div>

### <center>**DISUSUN OLEH :**</center>

#### <center>**DWI REZKY MAHARANI          (H061241078)**<center>

### <center>**PROGRAM STUDI GEOFISIKA**</center>
### <center>**FAKULTAS MATEMATIKA DAN ILMU PENGETAHUAN ALAM**</center>
#### <center>**UNIVERSITAS HASANUDDIN**</center>
#### <center>**MAKASSAR**</center>
#### <center>**2026**</center>

<hr>

# Form Prompt AI untuk Geofisika (Geolistrik dan AWS)


<div align="center">

| No | Bidang | Topik Prompt |
|----|--------|--------------|
| 1 | Geolistrik | Inversi 1D sounding Schlumberger (VES) dan penentuan lapisan akuifer |
| 2 | Geolistrik | Interpretasi penampang ERT 2D untuk dugaan bidang gelincir longsor |
| 3 | AWS | QA/QC data Automatic Weather Station 10 menitan |
| 4 | AWS | Analisis pola diurnal dan variasi bulanan suhu dan curah hujan |
| 5 | Geolistrik + AWS | Integrasi curah hujan AWS dan ERT time-lapse untuk infiltrasi |

</div>

<br>
<br>

<div align="center">

| Kode | Komponen | Fungsi |
|------|----------|--------|
| **R** | Role | Sudut pandang kerja, bukan credential |
| **T** | Task | Verb yang bisa diperiksa + objek + domain + kondisi penerimaan |
| **C** | Context | Tujuan, audiens, workflow, keputusan yang didukung |
| **I** | Input | Path, format, variabel, unit, CRS/datum, provenance |
| **K** | Constraints | File yang tidak boleh diubah, method, package, jaringan, approval |
| **O** | Output | Kontrak output: path, kolom/unit, format (rencana teks, belum hasil) |
| **Q** | Quality criteria | Kriteria yang dapat dinilai eksplisit |
| **V** | Optional verification | Check + expected + tindakan bila gagal |

</div>

> Bagian **O** adalah *expected output*, bukan *actual result*. Output berupa narasi dan tabel file; gambar belum dibuat, hanya direncanakan dalam teks.

## Template Umum

```markdown
## [R] Role
Bertindaklah sebagai <sudut pandang kerja>. Fokus pada <apa yang diperiksa>.

## [T] Task
<Verb> <objek> <domain/kondisi>, lalu <verb kedua>. <Kondisi penerimaan>.

## [C] Context
<Tujuan>, <audiens>, <tahap workflow>. Hasil dipakai untuk <keputusan>, bukan untuk <hal yang tidak dilakukan>.

## [I] Input
- `<path>`: format, variabel/kolom, unit, CRS/datum, sumber

## [K] Constraints
- Jangan mengubah <file/folder>.
- Method: <yang boleh>; <yang dilarang>.
- Software: <yang tersedia>; tanpa jaringan.
- Bila <kondisi tak jelas>: berhenti dan tanyakan.

## [O] Output (rencana teks)
- `<path output>`: <isi, kolom, unit>

## [Q] Quality criteria
- <kriteria terukur>
## [V] Optional verification
- Check: <command>. Expected: <kondisi>. Jika gagal: <tindakan>.
```

## **Prompt 1 — Geolistrik: Inversi 1D Sounding Schlumberger (VES) dan Penentuan Akuifer**

### **[R] Role**
Bertindaklah sebagai reviewer interpretasi geolistrik untuk hidrogeologi. Fokus pada kualitas data sounding, transparansi asumsi model lapisan, dan pemisahan antara hasil inversi dan dugaan litologi.

### **[T] Task**
Hitung resistivitas semu dari data sounding Schlumberger `data/raw/ves_points.csv`, lakukan inversi 1D per titik sounding, lalu susun tabel model lapisan (ketebalan dan resistivitas) dan narasi dugaan lapisan akuifer. Jangan menetapkan titik pengeboran final.

### **[C] Context**
Survei VES dilakukan di `{{lokasi}}` pada litologi dominan `{{aluvial/batupasir/vulkanik}}` untuk mendukung rencana sumur air tanah `{{desa/irigasi}}`. Hasil dipakai mahasiswa dan tim lapangan sebagai bahan diskusi awal. Hasil tidak dipakai sebagai keputusan pengeboran tanpa korelasi data bor.

### **[I] Input**
- `data/raw/ves_points.csv`: kolom `point_id`, `AB2_m` (m), `MN2_m` (m), `I_mA` (mA), `dV_mV` (mV), `lat`, `lon` (WGS84). Sumber: resistivitymeter `{{merek/model}}`.
- Konfigurasi: Schlumberger, faktor geometri `K = π (L² − l²) / (2l)` dengan `L = AB/2` dan `l = MN/2`.
- Parameter inversi: jumlah lapisan awal `{{3/4}}`, batas resistivitas `{{min–max Ω·m}}`.
- Data pembanding (bila ada): `data/raw/borelog.csv` (`depth_m`, `litologi`).

### **[K] Constraints**
- Jangan mengubah `data/raw/`; jangan menghapus titik atau baris yang janggal, cukup beri flag.
- Jangan mengarang data atau mengisi nilai hilang dengan interpolasi.
- Semua asumsi (jumlah lapisan, model awal, batas resistivitas) ditulis eksplisit di `run_config.json`.
- Software: Python dengan NumPy dan SciPy; pyGIMLi/ResIPy hanya bila sudah terpasang; tanpa jaringan.
- Kesimpulan hidrogeologi dinyatakan sebagai indikasi, bukan kepastian; jangan menyebut kedalaman bor final.
- Bila jumlah lapisan tidak dapat dibenarkan oleh data: berhenti dan tanyakan.

### **[O] Output (rencana teks)**
- `results/ves/ves_apparent_rho.csv`: `point_id`, `AB2_m`, `K_m`, `rho_a_ohm_m`, `qc_flag`.
- `results/ves/ves_layers.csv`: `point_id`, `layer_no`, `depth_top_m`, `depth_bottom_m`, `thickness_m`, `resistivity_ohm_m`, `rms_pct`.
- `results/ves/interpretation.md`: ringkasan kualitas data, penjelasan tiap lapisan dan dugaan litologi, kandidat lapisan akuifer beserta rentang kedalaman, daftar asumsi dan keterbatasan. Grafik hanya direncanakan (kurva sounding dan model inversi), belum digambar.
- `results/ves/run_config.json`: jumlah lapisan, model awal, batas, versi package.

### **[Q] Quality criteria**
- RMS error inversi dilaporkan untuk setiap titik.
- Setiap klaim litologi disertai nilai resistivitas dan dibandingkan dengan rentang umum (mis. pasir jenuh air vs lempung).
- Ketidakpastian (prinsip ekuivalensi) disebutkan.
- Jumlah titik output = jumlah titik input (termasuk yang di-flag).
- Fakta terukur, asumsi, dan dugaan dipisahkan dalam laporan.

### **[V] Optional verification**
- Check: hitung SHA-256 `ves_points.csv` sebelum dan sesudah prosedur. Expected: identik. Jika gagal: hentikan dan laporkan.
- Check: ulangi inversi dengan 3 dan 4 lapisan pada 2 titik. Expected: lapisan akuifer utama tetap pada kedalaman serupa. Jika tidak: laporkan sebagai model tidak stabil.
- Check: bandingkan dengan `borelog.csv` bila ada. Expected: dugaan litologi konsisten. Jika tidak: catat ketidaksesuaian, jangan menyesuaikan model.

In [7]:
import os, sys, json, hashlib, platform
import numpy as np
import pandas as pd
import scipy
from scipy.special import j1

# ----------------------------- PARAMETER (isi sesuai prompt) -----------------
LOKASI        = "Makassar, Sulawesi Selatan (data simulasi)"
LITOLOGI      = "aluvial"
N_LAYERS      = 3                  # jumlah lapisan awal {{3/4}}
RHO_MIN       = 1.0                # batas resistivitas (ohm.m)
RHO_MAX       = 1000.0
H_MIN         = 0.3                # batas ketebalan lapisan (m)
JUMP_LOG10    = 0.5                # ambang lonjakan antar titik AB/2 (selisih log10 rho_a)
AQ_RHO        = (30.0, 200.0)      # rentang resistivitas kandidat akuifer (ohm.m) - asumsi
MAX_ITER      = 60
VERIFIKASI_JML_LAPISAN = True      # ulangi inversi 3 & 4 lapisan pada 2 titik
BUAT_DATA_SINTETIS = True          # True: bila ves_points.csv belum ada, dibuat data SIMULASI otomatis

PATH_VES      = "data/raw/ves_points.csv"
PATH_BORELOG  = "data/raw/borelog.csv"
OUT_DIR       = "results/ves"

# ----------------------------- DATA SINTETIS (bila file belum ada) -----------
# PERHATIAN: ini data simulasi untuk uji skrip, BUKAN data lapangan.
if (not os.path.exists(PATH_VES)) and BUAT_DATA_SINTETIS:
    os.makedirs(os.path.dirname(PATH_VES), exist_ok=True)
    lam_s = np.logspace(-4, 2.5, 12000)
    dl_s = np.log(lam_s[1] / lam_s[0])
    model_s = {"VES1": ([40, 8, 120], [3, 12]),
               "VES2": ([60, 150, 15], [2, 20]),
               "VES3": ([30, 90, 10], [4, 15])}
    AB2_s = np.array([1, 1.5, 2, 3, 4, 6, 8, 10, 15, 20, 30, 40, 60, 80, 100.0])
    rng_s = np.random.default_rng(1)
    baris_s = []
    for pid_s in model_s:
        rho_s, h_s = model_s[pid_s]
        MN2_s = np.where(AB2_s < 10, 0.5, 5.0)
        T_s = np.full_like(lam_s, rho_s[-1])
        for i_s in range(1, -1, -1):
            th_s = np.tanh(lam_s * h_s[i_s])
            T_s = (T_s + rho_s[i_s] * th_s) / (1 + T_s * th_s / rho_s[i_s])
        ra_s = rho_s[0] + AB2_s**2 * ((T_s - rho_s[0])[None, :] * j1(lam_s[None, :] * AB2_s[:, None]) * lam_s**2 * dl_s).sum(axis=1)
        ra_s = ra_s * np.exp(rng_s.normal(0, 0.02, len(AB2_s)))
        K_s = np.pi * (AB2_s**2 - MN2_s**2) / (2 * MN2_s)
        for a_s in range(len(AB2_s)):
            baris_s.append([pid_s, AB2_s[a_s], MN2_s[a_s], 100.0, ra_s[a_s] / K_s[a_s] * 100.0, -5.1, 119.5])
    pd.DataFrame(baris_s, columns=["point_id", "AB2_m", "MN2_m", "I_mA", "dV_mV", "lat", "lon"]).to_csv(PATH_VES, index=False)

if not os.path.exists(PATH_VES):
    sys.exit("File tidak ditemukan: %s\nFolder kerja saat ini: %s\nLetakkan ves_points.csv di sana atau set BUAT_DATA_SINTETIS = True." % (PATH_VES, os.getcwd()))

# ----------------------------- CEK SHA-256 SEBELUM ---------------------------
f = open(PATH_VES, "rb"); sha_sebelum = hashlib.sha256(f.read()).hexdigest(); f.close()
os.makedirs(OUT_DIR, exist_ok=True)

# ----------------------------- BACA DATA -------------------------------------
df = pd.read_csv(PATH_VES)
kolom_wajib = ["point_id", "AB2_m", "MN2_m", "I_mA", "dV_mV", "lat", "lon"]
for k in kolom_wajib:
    if k not in df.columns:
        sys.exit("Kolom wajib tidak ada: " + k)
df["point_id"] = df["point_id"].astype(str)

# ----------------------------- RESISTIVITAS SEMU -----------------------------
# K = pi (L^2 - l^2) / (2 l), L = AB/2, l = MN/2 ; rho_a = K * dV / I
L_ = df["AB2_m"].astype(float)
l_ = df["MN2_m"].astype(float)
df["K_m"] = np.pi * (L_**2 - l_**2) / (2 * l_)
df["rho_a_ohm_m"] = df["K_m"] * (df["dV_mV"] / df["I_mA"])     # mV/mA = ohm

# ----------------------------- QC FLAG (tidak ada baris dihapus) -------------
flags = [[] for _ in range(len(df))]
for i in range(len(df)):
    r = df.iloc[i]
    nilai = [r["AB2_m"], r["MN2_m"], r["I_mA"], r["dV_mV"]]
    if any(pd.isna(nilai)):
        flags[i].append("data_hilang")
    elif r["MN2_m"] >= r["AB2_m"] or r["AB2_m"] <= 0 or r["MN2_m"] <= 0:
        flags[i].append("geometri_tidak_valid")
    elif r["I_mA"] <= 0:
        flags[i].append("arus_nonpositif")
    elif (not np.isfinite(r["rho_a_ohm_m"])) or r["rho_a_ohm_m"] <= 0:
        flags[i].append("rho_a_nonpositif")
FLAG_FATAL = ["data_hilang", "geometri_tidak_valid", "arus_nonpositif", "rho_a_nonpositif"]

# lonjakan antar titik AB/2 berurutan (hanya flag, data tetap dipakai)
for pid in df["point_id"].unique():
    idx = df.index[(df["point_id"] == pid)].tolist()
    idx = df.loc[idx].sort_values("AB2_m").index.tolist()
    ok = [j for j in idx if not any(x in FLAG_FATAL for x in flags[j])]
    for a in range(1, len(ok)):
        d = abs(np.log10(df.loc[ok[a], "rho_a_ohm_m"]) - np.log10(df.loc[ok[a-1], "rho_a_ohm_m"]))
        if d > JUMP_LOG10:
            flags[ok[a]].append("lonjakan")
df["qc_flag"] = [";".join(x) if len(x) > 0 else "ok" for x in flags]

df_rho = df[["point_id", "AB2_m", "K_m", "rho_a_ohm_m", "qc_flag"]]
df_rho.to_csv(os.path.join(OUT_DIR, "ves_apparent_rho.csv"), index=False)

# ----------------------------- DAFTAR PEKERJAAN INVERSI ----------------------
titik = list(dict.fromkeys(df["point_id"].tolist()))
jobs = [(p, N_LAYERS, "utama") for p in titik]
if VERIFIKASI_JML_LAPISAN:
    for p in titik[:2]:
        for nl in (3, 4):
            jobs.append((p, nl, "verifikasi"))

# ----------------------------- GRID BILANGAN GELOMBANG -----------------------
NLAM = 12000
lam = np.logspace(-4, 2.5, NLAM)            # 1/m
dlnlam = np.log(lam[1] / lam[0])

hasil_lapisan = []     # baris ves_layers.csv (utama)
hasil_verif = []
detail_titik = {}      # untuk interpretasi

for (pid, NL, tag) in jobs:
    d = df[(df["point_id"] == pid)]
    d = d[[not any(x in FLAG_FATAL for x in flags[j]) for j in d.index]]
    d = d.sort_values("AB2_m")
    obs = d["rho_a_ohm_m"].values.astype(float)
    L = d["AB2_m"].values.astype(float)
    npar = 2 * NL - 1

    # jumlah lapisan harus bisa dibenarkan data: n data > n parameter
    if len(obs) < npar + 1:
        if tag == "utama":
            sys.exit("BERHENTI: titik %s hanya punya %d data valid, tidak cukup untuk %d lapisan "
                     "(%d parameter). Kurangi jumlah lapisan atau tambah data. Mohon konfirmasi."
                     % (pid, len(obs), NL, npar))
        else:
            continue

    # matriks kernel Hankel: (nL, nlam)
    J = j1(lam[None, :] * L[:, None]) * (lam ** 2)[None, :] * dlnlam

    # model awal (log10)
    kel = np.array_split(np.log10(obs), NL)
    rho0 = [np.mean(g) for g in kel]
    h0 = np.log10(np.geomspace(max(L.min(), H_MIN * 2), L.max() / 3.0, NL - 1))
    p = np.r_[rho0, h0]
    p = np.clip(p, np.r_[np.full(NL, np.log10(RHO_MIN)), np.full(NL - 1, np.log10(H_MIN))],
                   np.r_[np.full(NL, np.log10(RHO_MAX)), np.full(NL - 1, np.log10(L.max()))])
    lo = np.r_[np.full(NL, np.log10(RHO_MIN)), np.full(NL - 1, np.log10(H_MIN))]
    hi = np.r_[np.full(NL, np.log10(RHO_MAX)), np.full(NL - 1, np.log10(L.max()))]
    model_awal = (10 ** p).tolist()

    mus = np.array([1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0, 1e3])
    eps = 1e-4
    stage = "jac"
    it = 0
    cost = None
    Jm = None; rres = None; trials = None
    selesai = False
    while not selesai:
        # ---- 1 blok forward Pekeris/Hankel (dipakai bergantian untuk Jacobian & trial) ----
        if stage == "jac":
            PB = np.vstack([p, p[None, :] + eps * np.eye(npar)])
        else:
            PB = trials
        rho_b = 10 ** PB[:, :NL]
        h_b = 10 ** PB[:, NL:]
        T = np.repeat(rho_b[:, NL-1:NL], NLAM, axis=1)
        for i in range(NL - 2, -1, -1):
            th = np.tanh(lam[None, :] * h_b[:, i:i+1])
            r_i = rho_b[:, i:i+1]
            T = (T + r_i * th) / (1.0 + T * th / r_i)
        dT = T - rho_b[:, 0:1]
        rho_calc = rho_b[:, 0:1] + (L[None, :] ** 2) * (dT @ J.T)
        rho_calc = np.maximum(rho_calc, 1e-6)
        logF = np.log10(rho_calc)
        # -----------------------------------------------------------------------------------
        if stage == "jac":
            rres = np.log10(obs) - logF[0]
            cost = float(np.sum(rres ** 2))
            Jm = ((logF[1:] - logF[0][None, :]) / eps).T          # (nL, npar)
            A = Jm.T @ Jm
            g = Jm.T @ rres
            trials = []
            for mu in mus:
                step = np.linalg.solve(A + mu * np.diag(np.diag(A) + 1e-9), g)
                trials.append(np.clip(p + step, lo, hi))
            trials = np.array(trials)
            stage = "trial"
        else:
            costs = np.sum((np.log10(obs)[None, :] - logF) ** 2, axis=1)
            k = int(np.argmin(costs))
            it += 1
            if costs[k] < cost:
                rel = (cost - costs[k]) / max(cost, 1e-12)
                p = trials[k].copy()
                if rel < 1e-6 or it >= MAX_ITER:
                    selesai = True
                stage = "jac"
            else:
                selesai = True

    # ---- hitung ulang respons akhir & RMS (ulang forward sekali, p final) ----
    PB = p[None, :]
    rho_b = 10 ** PB[:, :NL]; h_b = 10 ** PB[:, NL:]
    T = np.repeat(rho_b[:, NL-1:NL], NLAM, axis=1)
    for i in range(NL - 2, -1, -1):
        th = np.tanh(lam[None, :] * h_b[:, i:i+1]); r_i = rho_b[:, i:i+1]
        T = (T + r_i * th) / (1.0 + T * th / r_i)
    calc = (rho_b[:, 0:1] + (L[None, :] ** 2) * ((T - rho_b[:, 0:1]) @ J.T))[0]
    rms_pct = float(np.sqrt(np.mean(((obs - calc) / obs) ** 2)) * 100.0)

    # ---- ketidakpastian (ekuivalensi) dari matriks sensitivitas ----
    dof = max(len(obs) - npar, 1)
    s2 = float(np.sum((np.log10(obs) - np.log10(calc)) ** 2)) / dof
    cov = np.linalg.pinv(Jm.T @ Jm) * s2
    sd = np.sqrt(np.abs(np.diag(cov)))
    faktor_rho = 10 ** sd[:NL]

    rho_fin = 10 ** p[:NL]
    h_fin = 10 ** p[NL:]
    top = np.r_[0.0, np.cumsum(h_fin)]
    bot = np.r_[np.cumsum(h_fin), np.nan]
    baris = []
    for i in range(NL):
        baris.append({
            "point_id": pid, "layer_no": i + 1,
            "depth_top_m": round(float(top[i]), 3),
            "depth_bottom_m": (round(float(bot[i]), 3) if i < NL - 1 else np.nan),
            "thickness_m": (round(float(h_fin[i]), 3) if i < NL - 1 else np.nan),
            "resistivity_ohm_m": round(float(rho_fin[i]), 2),
            "rms_pct": round(rms_pct, 2)})
    if tag == "utama":
        hasil_lapisan += baris
        detail_titik[pid] = {"rms": rms_pct, "faktor_rho": faktor_rho, "n_data": len(obs), "iter": it}
    else:
        hasil_verif.append({"point_id": pid, "NL": NL, "baris": baris, "rms": rms_pct})

# titik yang tidak terinversi tetap dicatat (jumlah titik output = input)
sudah = set([r["point_id"] for r in hasil_lapisan])
for pid in titik:
    if pid not in sudah:
        hasil_lapisan.append({"point_id": pid, "layer_no": np.nan, "depth_top_m": np.nan,
                              "depth_bottom_m": np.nan, "thickness_m": np.nan,
                              "resistivity_ohm_m": np.nan, "rms_pct": np.nan})
df_lay = pd.DataFrame(hasil_lapisan)
df_lay.to_csv(os.path.join(OUT_DIR, "ves_layers.csv"), index=False)

# ----------------------------- INTERPRETASI (Markdown) -----------------------
borelog = None
if os.path.exists(PATH_BORELOG):
    borelog = pd.read_csv(PATH_BORELOG)

tabel_lito = [(0, 10, "lempung / material jenuh air asin (indikasi)"),
              (10, 50, "lempung pasiran atau pasir lempungan jenuh (indikasi)"),
              (50, 200, "pasir-kerikil jenuh air tawar (indikasi)"),
              (200, 1e12, "material kering / batuan kompak (indikasi)")]

md = []
md.append("# Interpretasi VES Schlumberger\n")
md.append("Lokasi: %s | Litologi dominan: %s\n" % (LOKASI, LITOLOGI))
md.append("> Hasil ini indikasi awal untuk diskusi. **Bukan** keputusan titik/kedalaman bor final.\n")
md.append("## 1. Fakta terukur (data)\n")
md.append("- Jumlah titik input: %d; jumlah baris data: %d." % (len(titik), len(df)))
for q in sorted(set([x for f_ in flags for x in f_])):
    md.append("- Baris dengan flag `%s`: %d" % (q, sum([1 for f_ in flags if q in f_])))
md.append("- Titik tanpa flag: %d baris `ok`.\n" % sum([1 for f_ in flags if len(f_) == 0]))
md.append("## 2. Hasil inversi per titik (RMS dan lapisan)\n")
kandidat = []
for pid in titik:
    sub = df_lay[df_lay["point_id"] == pid]
    md.append("### Titik %s" % pid)
    if sub["layer_no"].isna().all():
        md.append("Tidak diinversi (data valid tidak cukup).\n"); continue
    md.append("RMS inversi = **%.2f %%** (n data = %d, iterasi = %d)\n" %
              (detail_titik[pid]["rms"], detail_titik[pid]["n_data"], detail_titik[pid]["iter"]))
    md.append("| Lapisan | Kedalaman (m) | Tebal (m) | rho (ohm.m) | faktor ketidakpastian rho | Dugaan litologi |")
    md.append("|---|---|---|---|---|---|")
    for _, r in sub.iterrows():
        lito = [t[2] for t in tabel_lito if t[0] <= r["resistivity_ohm_m"] < t[1]][0]
        kd = "%.1f - %s" % (r["depth_top_m"], ("%.1f" % r["depth_bottom_m"]) if not np.isnan(r["depth_bottom_m"]) else "tak hingga")
        tb = ("%.1f" % r["thickness_m"]) if not np.isnan(r["thickness_m"]) else "-"
        fk = detail_titik[pid]["faktor_rho"][int(r["layer_no"]) - 1]
        md.append("| %d | %s | %s | %.1f | x/÷ %.2f | %s |" % (r["layer_no"], kd, tb, r["resistivity_ohm_m"], fk, lito))
        if r["layer_no"] > 1 and AQ_RHO[0] <= r["resistivity_ohm_m"] <= AQ_RHO[1]:
            kandidat.append((pid, int(r["layer_no"]), r["depth_top_m"], r["depth_bottom_m"], r["resistivity_ohm_m"]))
    md.append("")
md.append("## 3. Kandidat lapisan akuifer (dugaan)\n")
md.append("Kriteria: lapisan bukan lapisan teratas dengan rho %.0f-%.0f ohm.m (asumsi, bandingkan rentang umum pasir jenuh vs lempung).\n" % AQ_RHO)
if len(kandidat) == 0:
    md.append("Tidak ada lapisan yang memenuhi kriteria.\n")
for k in kandidat:
    bt = "tak hingga (ketebalan tak terjamin)" if np.isnan(k[3]) else "%.1f m" % k[3]
    md.append("- Titik %s, lapisan %d: kedalaman %.1f - %s, rho = %.1f ohm.m" % (k[0], k[1], k[2], bt, k[4]))
    if borelog is not None:
        md.append("  - (pembanding borelog tersedia: periksa manual litologi pada rentang kedalaman ini)")
md.append("")
md.append("## 4. Asumsi dan keterbatasan\n")
md.append("- Model %d lapisan, model awal dari kuantil data, batas rho %.0f-%.0f ohm.m, tebal min %.1f m." % (N_LAYERS, RHO_MIN, RHO_MAX, H_MIN))
md.append("- **Prinsip ekuivalensi**: kombinasi tebal dan rho berbeda dapat memberi kurva yang hampir sama; kolom faktor ketidakpastian hanya perkiraan linear.")
md.append("- Lapisan terbawah dianggap half-space; ketebalannya tidak terukur.")
md.append("- Rentang rho untuk litologi bersifat umum dan tumpang tindih; wajib dikorelasikan dengan data bor.")
md.append("- Grafik (kurva sounding dan model inversi) hanya direncanakan, belum dibuat.\n")
if len(hasil_verif) > 0:
    md.append("## 5. Uji kestabilan jumlah lapisan (3 vs 4 lapisan, 2 titik)\n")
    for v in hasil_verif:
        s = "; ".join(["L%d rho=%.0f top=%.1f" % (b["layer_no"], b["resistivity_ohm_m"], b["depth_top_m"]) for b in v["baris"]])
        md.append("- Titik %s, %d lapisan, RMS %.2f %%: %s" % (v["point_id"], v["NL"], v["rms"], s))
    md.append("\nBandingkan manual: bila kedalaman akuifer utama berbeda jauh antar jumlah lapisan, laporkan model **tidak stabil**.\n")
f = open(os.path.join(OUT_DIR, "interpretation.md"), "w", encoding="utf-8"); f.write("\n".join(md)); f.close()

# ----------------------------- CEK SHA-256 SESUDAH ---------------------------
f = open(PATH_VES, "rb"); sha_sesudah = hashlib.sha256(f.read()).hexdigest(); f.close()
if sha_sebelum != sha_sesudah:
    sys.exit("GAGAL: ves_points.csv berubah selama proses!")

# ----------------------------- RUN CONFIG ------------------------------------
cfg = {"n_layers": N_LAYERS, "rho_min": RHO_MIN, "rho_max": RHO_MAX, "h_min_m": H_MIN,
       "jump_log10": JUMP_LOG10, "aquifer_rho_range": AQ_RHO, "max_iter": MAX_ITER,
       "model_awal": "rho: rata-rata log10 data per kelompok AB/2; tebal: deret geometri",
       "metode": "Levenberg-Marquardt (log10), forward Pekeris + integral Hankel J1",
       "sha256_ves_points": sha_sesudah, "sha256_identik": sha_sebelum == sha_sesudah,
       "versi": {"python": platform.python_version(), "numpy": np.__version__,
                 "scipy": scipy.__version__, "pandas": pd.__version__}}
f = open(os.path.join(OUT_DIR, "run_config.json"), "w"); json.dump(cfg, f, indent=2); f.close()

# ----------------------------- HASIL AKHIR -----------------------------------
print("VES selesai: rhoa dan QC disimpan di", os.path.abspath(OUT_DIR))

VES selesai: rhoa dan QC disimpan di c:\Users\X270\Downloads\results\ves


## **Prompt 2 — Geolistrik: Interpretasi ERT 2D untuk Dugaan Bidang Gelincir**

### **[R] Role**
Bertindaklah sebagai reviewer interpretasi ERT untuk kebencanaan. Fokus pada pembedaan fakta data, interpretasi, dan alternatif penjelasan sebelum hasil dibaca pengambil keputusan.

### **[T] Task**
Deskripsikan sebaran resistivitas pada penampang hasil inversi `data/processed/ert_inverted.csv`, identifikasi zona resistivitas rendah, menengah, dan tinggi, lalu tandai zona yang diduga bidang gelincir atau lapisan jenuh air. Jangan menetapkan tingkat bahaya longsor.

### **[C] Context**
Lereng di `{{lokasi}}` mengalami retakan setelah hujan lebat. Survei memakai konfigurasi `{{dipole-dipole/Wenner}}`, spasi elektroda `{{5}}` m, panjang lintasan `{{n}}` m. Laporan dipakai sebagai bahan awal pemerintah daerah untuk menentukan survei lanjutan. Hasil bukan analisis stabilitas lereng atau peta zona bahaya.

### **[I] Input**
- `data/processed/ert_inverted.csv`: `x_m`, `z_m` (m, z positif ke atas, datum lokal), `rho_ohm_m` (Ω·m), `rms_pct`, `doi_flag` (bila ada).
- `data/processed/topography.csv`: `x_m`, `elev_m`.
- `data/raw/field_notes.md`: kondisi lapangan, posisi retakan, curah hujan sebelum survei `{{mm}}`.

### **[K] Constraints**
- Jangan mengubah `data/processed/` dan `data/raw/`.
- Jangan menjalankan ulang inversi atau mengubah regularisasi.
- Bedakan dengan jelas fakta dari data dan interpretasi; setiap zona wajib punya rentang Ω·m dan koordinat.
- Sebutkan bila kedalaman investigasi (DOI) tidak mendukung kesimpulan.
- Ringkasan akhir memakai bahasa yang dapat dipahami non-geofisikawan.
- Bila topografi tidak konsisten dengan penampang: berhenti dan tanyakan.

### **[O] Output (rencana teks)**
- `results/ert/zones.csv`: `zone_id`, `x_min_m`, `x_max_m`, `z_min_m`, `z_max_m`, `rho_min`, `rho_max` (Ω·m), `kelas` (rendah/menengah/tinggi), `dugaan_material`, `keyakinan`.
- `results/ert/interpretation.md`: deskripsi kiri ke kanan lintasan, penjelasan zona, posisi dan kedalaman dugaan bidang gelincir, alternatif interpretasi (lempung vs air), ringkasan untuk pengambil keputusan, saran survei lanjutan. Penampang hanya direncanakan dalam teks, belum digambar.
- `results/ert/run_config.json`: batas kelas resistivitas dan alasan pemilihannya.

### **[Q] Quality criteria**
- Batas kelas resistivitas dinyatakan dan diberi alasan.
- Interpretasi konsisten dengan topografi dan catatan lapangan.
- Minimal satu alternatif interpretasi dibahas untuk setiap zona penting.
- Zona di luar DOI ditandai tidak dapat diinterpretasi.
- Tidak ada pernyataan tingkat bahaya definitif.

### **[V] Optional verification**
- Check: hitung ulang rentang Ω·m dua zona langsung dari `ert_inverted.csv`. Expected: sama dengan `zones.csv`. Jika gagal: koreksi tabel dan laporkan.
- Check: bandingkan posisi dugaan bidang gelincir dengan lokasi retakan di `field_notes.md`. Expected: berdekatan atau dijelaskan bila tidak. Jika tidak: tulis sebagai ketidaksesuaian.

In [8]:
import os, sys, json, hashlib, platform, re
import numpy as np
import pandas as pd
import scipy
from scipy import ndimage
# ----------------------------- PARAMETER (isi sesuai prompt) -----------------
LOKASI         = "Makassar, Sulawesi Selatan (data simulasi)"
KONFIGURASI    = "dipole-dipole"        # {{dipole-dipole/Wenner}}
SPASI_M        = 5.0                    # spasi elektroda (m)
RHO_RENDAH_MAX = 100.0                  # < 100 ohm.m  -> kelas rendah   (asumsi)
RHO_TINGGI_MIN = 500.0                  # >= 500 ohm.m -> kelas tinggi   (asumsi)
MIN_CELL       = 6                      # zona lebih kecil dari ini diabaikan
MIN_DEPTH_SLIP = 2.0                    # kedalaman rata-rata minimum dugaan bidang gelincir (m)
MIN_LEBAR_FRAC = 0.15                   # lebar minimum zona terhadap panjang lintasan
RMS_BAIK       = 5.0                    # RMS (%) untuk keyakinan "tinggi"
RMS_BATAS      = 10.0                   # RMS (%) batas keyakinan "sedang"
TOL_TOPO_M     = SPASI_M                # toleransi selisih permukaan penampang vs topografi (m)
BUAT_DATA_SINTETIS = True               # True: bila file input belum ada, dibuat data SIMULASI

PATH_ERT   = "data/processed/ert_inverted.csv"
PATH_TOPO  = "data/processed/topography.csv"
PATH_NOTES = "data/raw/field_notes.md"
OUT_DIR    = "results/ert"

# ----------------------------- DATA SINTETIS (bila file belum ada) -----------
# PERHATIAN: ini data simulasi untuk uji skrip, BUKAN data lapangan.
if (not os.path.exists(PATH_ERT)) and BUAT_DATA_SINTETIS:
    os.makedirs(os.path.dirname(PATH_ERT), exist_ok=True)
    os.makedirs(os.path.dirname(PATH_NOTES), exist_ok=True)
    rng_s = np.random.default_rng(2)
    xs_s = np.arange(0, 100.1, 2.5)
    elev_s = 60.0 - 0.2 * xs_s
    ds_s = np.arange(0, 22.6, 2.5)
    baris_s = []
    for x_s, el_s in zip(xs_s, elev_s):
        for d_s in ds_s:
            if d_s < 4:
                rho_s = 300.0
            elif d_s < 11 and 20 <= x_s <= 75:
                rho_s = 45.0
            elif d_s < 11:
                rho_s = 250.0
            else:
                rho_s = 700.0
            rho_s = rho_s * np.exp(rng_s.normal(0, 0.05))
            baris_s.append([x_s, el_s - d_s, rho_s, 4.5, 1 if d_s > 17.5 else 0])
    pd.DataFrame(baris_s, columns=["x_m", "z_m", "rho_ohm_m", "rms_pct", "doi_flag"]).to_csv(PATH_ERT, index=False)
    pd.DataFrame({"x_m": np.arange(0, 100.1, 5.0), "elev_m": 60.0 - 0.2 * np.arange(0, 100.1, 5.0)}).to_csv(PATH_TOPO, index=False)
    f = open(PATH_NOTES, "w", encoding="utf-8")
    f.write("# Catatan lapangan (SIMULASI)\n- Lereng retak setelah hujan lebat.\n- Posisi retakan (x_m): 35, 52\n- Curah hujan sebelum survei: 85 mm\n")
    f.close()

for p_ in [PATH_ERT, PATH_TOPO]:
    if not os.path.exists(p_):
        sys.exit("File tidak ditemukan: %s\nFolder kerja saat ini: %s" % (p_, os.getcwd()))

# ----------------------------- CEK SHA-256 SEBELUM ---------------------------
sha_sebelum = {}
for p_ in [PATH_ERT, PATH_TOPO, PATH_NOTES]:
    if os.path.exists(p_):
        f = open(p_, "rb"); sha_sebelum[p_] = hashlib.sha256(f.read()).hexdigest(); f.close()
os.makedirs(OUT_DIR, exist_ok=True)

# ----------------------------- BACA DATA -------------------------------------
df = pd.read_csv(PATH_ERT)
topo = pd.read_csv(PATH_TOPO).sort_values("x_m")
for k in ["x_m", "z_m", "rho_ohm_m", "rms_pct"]:
    if k not in df.columns:
        sys.exit("Kolom wajib tidak ada di ert_inverted.csv: " + k)
for k in ["x_m", "elev_m"]:
    if k not in topo.columns:
        sys.exit("Kolom wajib tidak ada di topography.csv: " + k)

catatan = ""
if os.path.exists(PATH_NOTES):
    f = open(PATH_NOTES, "r", encoding="utf-8"); catatan = f.read(); f.close()

# posisi retakan & curah hujan dari catatan lapangan
x_retakan = []
hujan_mm = None
for baris in catatan.splitlines():
    if "retakan" in baris.lower():
        x_retakan += [float(v) for v in re.findall(r"\d+(?:\.\d+)?", baris.split(":", 1)[-1])]
    if "curah hujan" in baris.lower() and hujan_mm is None:
        angka = re.findall(r"\d+(?:\.\d+)?", baris.split(":", 1)[-1])
        if len(angka) > 0:
            hujan_mm = float(angka[0])

# ----------------------------- DOI -------------------------------------------
if "doi_flag" in df.columns:
    luar_doi = df["doi_flag"].astype(str).str.strip().str.lower().isin(["1", "1.0", "true", "yes", "ya", "outside"]).values
    ada_doi = True
else:
    luar_doi = np.zeros(len(df), dtype=bool)
    ada_doi = False
valid = (~np.isnan(df["rho_ohm_m"].values)) & (df["rho_ohm_m"].values > 0) & (~luar_doi)

# ----------------------------- CEK KONSISTENSI TOPOGRAFI ---------------------
df["zsurf"] = df.groupby("x_m")["z_m"].transform("max")
xu = np.sort(df["x_m"].unique())
zs_u = df.groupby("x_m")["z_m"].max().loc[xu].values
if xu.min() < topo["x_m"].min() - 1e-6 or xu.max() > topo["x_m"].max() + 1e-6:
    sys.exit("BERHENTI: rentang x penampang di luar rentang topografi. Mohon konfirmasi data topografi.")
elev_i = np.interp(xu, topo["x_m"].values, topo["elev_m"].values)
selisih = np.abs(zs_u - elev_i)
if selisih.max() > TOL_TOPO_M:
    sys.exit("BERHENTI: topografi tidak konsisten dengan penampang (selisih maksimum %.2f m di x = %.1f m, toleransi %.1f m). "
             "Mohon konfirmasi datum/elevasi." % (selisih.max(), xu[int(np.argmax(selisih))], TOL_TOPO_M))

# ----------------------------- GRID (x, kedalaman dari permukaan) ------------
df["depth_m"] = (df["zsurf"] - df["z_m"]).round(3)
ds = np.sort(df["depth_m"].unique())
ix = np.searchsorted(xu, df["x_m"].values)
idp = np.searchsorted(ds, df["depth_m"].values)
rho_mat = np.full((len(ds), len(xu)), np.nan)
z_mat = np.full((len(ds), len(xu)), np.nan)
cls_mat = np.full((len(ds), len(xu)), -1, dtype=int)
doi_mat = np.zeros((len(ds), len(xu)), dtype=bool)
rho_v = df["rho_ohm_m"].values
z_v = df["z_m"].values
for n in range(len(df)):
    rho_mat[idp[n], ix[n]] = rho_v[n]
    z_mat[idp[n], ix[n]] = z_v[n]
    doi_mat[idp[n], ix[n]] = luar_doi[n]
    if valid[n]:
        if rho_v[n] < RHO_RENDAH_MAX:
            cls_mat[idp[n], ix[n]] = 0
        elif rho_v[n] < RHO_TINGGI_MIN:
            cls_mat[idp[n], ix[n]] = 1
        else:
            cls_mat[idp[n], ix[n]] = 2

panjang_m = float(xu.max() - xu.min())
rms_rata = float(np.nanmean(df["rms_pct"].values))
nama_kelas = ["rendah", "menengah", "tinggi"]

# ----------------------------- IDENTIFIKASI ZONA -----------------------------
zona = []          # baris zones.csv
zona_sel = {}      # sel (x_m, z_m) tiap zona untuk verifikasi
zona_info = {}     # info tambahan untuk interpretasi
nomor = 0
kandidat = []
for k in range(3):
    lab, nlab = ndimage.label(cls_mat == k)
    for i in range(1, nlab + 1):
        idx = np.argwhere(lab == i)
        if len(idx) < MIN_CELL:
            continue
        nomor += 1
        zid = "Z%d" % nomor
        rr = idx[:, 0]; cc = idx[:, 1]
        xz = xu[cc]; zz = z_mat[rr, cc]; rh = rho_mat[rr, cc]
        depth_mean = float(ds[rr].mean())
        # apakah zona berada di bawah material lebih resistif (kolom yang sama)?
        n_bawah = 0
        kolom = np.unique(cc)
        for c in kolom:
            r0 = rr[cc == c].min()
            atas = rho_mat[:r0, c]
            atas = atas[(~np.isnan(atas)) & (~doi_mat[:r0, c])]
            if len(atas) > 0 and np.any(atas >= RHO_RENDAH_MAX):
                n_bawah += 1
        frac_bawah = n_bawah / len(kolom)
        lebar = float(xz.max() - xz.min())
        if k == 0:
            if frac_bawah >= 0.5 and depth_mean >= MIN_DEPTH_SLIP and lebar >= MIN_LEBAR_FRAC * panjang_m:
                material = "lapisan jenuh air atau lempung; DUGAAN bidang gelincir (terletak di bawah material lebih resistif)"
                kandidat.append(zid)
            else:
                material = "lempung/tanah lapuk jenuh atau zona basah dangkal (dugaan)"
        elif k == 1:
            material = "tanah lapuk atau pasir lempungan lembap (dugaan)"
        else:
            material = "batuan kompak/material kering, atau rongga bila terisolasi (dugaan)"
        if len(idx) >= 60 and rms_rata <= RMS_BAIK:
            yakin = "tinggi"
        elif len(idx) >= 30 and rms_rata <= RMS_BATAS:
            yakin = "sedang"
        else:
            yakin = "rendah"
        zona.append({"zone_id": zid, "x_min_m": round(float(xz.min()), 2), "x_max_m": round(float(xz.max()), 2),
                     "z_min_m": round(float(zz.min()), 2), "z_max_m": round(float(zz.max()), 2),
                     "rho_min": round(float(rh.min()), 1), "rho_max": round(float(rh.max()), 1),
                     "kelas": nama_kelas[k], "dugaan_material": material, "keyakinan": yakin})
        zona_sel[zid] = pd.DataFrame({"x_m": xz, "z_m": zz})
        zona_info[zid] = {"n_sel": len(idx), "depth_top": float(ds[rr].min()), "depth_bot": float(ds[rr].max()),
                          "depth_mean": depth_mean, "frac_bawah": frac_bawah}

# zona di luar DOI -> tidak dapat diinterpretasi
if luar_doi.any():
    xd = df["x_m"].values[luar_doi]; zd = df["z_m"].values[luar_doi]; rd = df["rho_ohm_m"].values[luar_doi]
    zona.append({"zone_id": "OUT-DOI", "x_min_m": round(float(xd.min()), 2), "x_max_m": round(float(xd.max()), 2),
                 "z_min_m": round(float(zd.min()), 2), "z_max_m": round(float(zd.max()), 2),
                 "rho_min": round(float(np.nanmin(rd)), 1), "rho_max": round(float(np.nanmax(rd)), 1),
                 "kelas": "tidak dapat diinterpretasi", "dugaan_material": "di luar kedalaman investigasi (DOI)",
                 "keyakinan": "-"})
df_zona = pd.DataFrame(zona)
df_zona.to_csv(os.path.join(OUT_DIR, "zones.csv"), index=False)

# ----------------------------- PERBANDINGAN DENGAN RETAKAN -------------------
banding = []
for zid in kandidat:
    r = df_zona[df_zona["zone_id"] == zid].iloc[0]
    dekat = [xr for xr in x_retakan if (r["x_min_m"] - SPASI_M) <= xr <= (r["x_max_m"] + SPASI_M)]
    banding.append((zid, r["x_min_m"], r["x_max_m"], dekat))

# ----------------------------- INTERPRETASI (Markdown) -----------------------
kelas_teks = np.array(["rendah", "menengah", "tinggi"])
md = []
md.append("# Interpretasi ERT 2D - Dugaan Bidang Gelincir\n")
md.append("Lokasi: %s | Konfigurasi: %s | Spasi elektroda: %.1f m | Panjang lintasan: %.1f m\n" % (LOKASI, KONFIGURASI, SPASI_M, panjang_m))
md.append("> Ini interpretasi awal untuk bahan survei lanjutan. **Bukan** analisis stabilitas lereng dan **bukan** penetapan tingkat bahaya longsor.\n")
md.append("## 1. Fakta dari data\n")
md.append("- Jumlah sel model: %d; sel di luar DOI: %d; sel dipakai untuk interpretasi: %d." % (len(df), int(luar_doi.sum()), int(valid.sum())))
md.append("- Rentang resistivitas (sel valid): %.1f - %.1f ohm.m." % (np.nanmin(rho_v[valid]), np.nanmax(rho_v[valid])))
md.append("- RMS inversi rata-rata: %.2f %%." % rms_rata)
md.append("- Topografi konsisten dengan penampang (selisih maksimum %.2f m, toleransi %.1f m)." % (selisih.max(), TOL_TOPO_M))
if not ada_doi:
    md.append("- **Peringatan**: kolom `doi_flag` tidak ada; batas DOI tidak dapat diverifikasi sehingga kedalaman terbawah harus dibaca hati-hati.")
if hujan_mm is not None:
    md.append("- Curah hujan sebelum survei (catatan lapangan): %.0f mm." % hujan_mm)
if len(x_retakan) > 0:
    md.append("- Posisi retakan (catatan lapangan): x = %s m." % ", ".join(["%.0f" % v for v in x_retakan]))
md.append("")
md.append("**Batas kelas resistivitas** (asumsi, bukan fakta data):\n")
md.append("- rendah: < %.0f ohm.m (rentang umum tanah lempung/jenuh air)" % RHO_RENDAH_MAX)
md.append("- menengah: %.0f - %.0f ohm.m (tanah lapuk, pasir lempungan)" % (RHO_RENDAH_MAX, RHO_TINGGI_MIN))
md.append("- tinggi: >= %.0f ohm.m (batuan kompak atau material kering)" % RHO_TINGGI_MIN)
md.append("- Rentang resistivitas tumpang tindih antar material; batas ini harus disesuaikan dengan data lokal bila tersedia.\n")

md.append("## 2. Deskripsi sebaran resistivitas (kiri ke kanan lintasan)\n")
edges = np.linspace(xu.min(), xu.max(), 6)
dmax = ds.max()
for s in range(5):
    kol = (xu >= edges[s]) & ((xu <= edges[s + 1]) if s == 4 else (xu < edges[s + 1]))
    rm = np.where(cls_mat[:, kol] >= 0, rho_mat[:, kol], np.nan)
    dangkal = rm[ds <= dmax / 3.0, :]
    dalam = rm[ds > dmax / 3.0, :]
    if np.all(np.isnan(dangkal)):
        txt_a = "tidak ada data valid"
    else:
        va = np.nanmean(dangkal); txt_a = "%.0f ohm.m (%s)" % (va, kelas_teks[0 if va < RHO_RENDAH_MAX else (1 if va < RHO_TINGGI_MIN else 2)])
    if np.all(np.isnan(dalam)):
        txt_b = "tidak ada data valid"
    else:
        vb = np.nanmean(dalam); txt_b = "%.0f ohm.m (%s)" % (vb, kelas_teks[0 if vb < RHO_RENDAH_MAX else (1 if vb < RHO_TINGGI_MIN else 2)])
    md.append("- x = %.0f - %.0f m: bagian atas (kedalaman <= %.1f m) rata-rata %s; bagian bawah rata-rata %s." % (edges[s], edges[s + 1], dmax / 3.0, txt_a, txt_b))
md.append("")

md.append("## 3. Zona resistivitas\n")
md.append("| Zona | x (m) | z (m, elevasi) | Kedalaman dari permukaan (m) | rho (ohm.m) | Kelas | Dugaan material | Keyakinan |")
md.append("|---|---|---|---|---|---|---|---|")
for r in zona:
    zi = zona_info.get(r["zone_id"])
    kd = ("%.1f - %.1f" % (zi["depth_top"], zi["depth_bot"])) if zi is not None else "-"
    md.append("| %s | %.1f - %.1f | %.1f - %.1f | %s | %.0f - %.0f | %s | %s | %s |" % (
        r["zone_id"], r["x_min_m"], r["x_max_m"], r["z_min_m"], r["z_max_m"], kd,
        r["rho_min"], r["rho_max"], r["kelas"], r["dugaan_material"], r["keyakinan"]))
md.append("\nZona **OUT-DOI** (bila ada) berada di luar kedalaman investigasi dan **tidak dapat diinterpretasi**.\n")

md.append("## 4. Dugaan bidang gelincir / lapisan jenuh air\n")
if len(kandidat) == 0:
    md.append("Tidak ada zona resistivitas rendah yang memenuhi kriteria (di bawah material lebih resistif, kedalaman rata-rata >= %.1f m, lebar >= %.0f %% lintasan).\n" % (MIN_DEPTH_SLIP, MIN_LEBAR_FRAC * 100))
for zid, xa, xb, dekat in banding:
    zi = zona_info[zid]; r = df_zona[df_zona["zone_id"] == zid].iloc[0]
    md.append("- **%s**: x = %.1f - %.1f m, kedalaman %.1f - %.1f m (rata-rata %.1f m), rho %.0f - %.0f ohm.m, keyakinan %s." % (
        zid, xa, xb, zi["depth_top"], zi["depth_bot"], zi["depth_mean"], r["rho_min"], r["rho_max"], r["keyakinan"]))
    if len(x_retakan) == 0:
        md.append("  - Posisi retakan tidak tersedia di catatan lapangan; tidak dapat dibandingkan.")
    elif len(dekat) > 0:
        md.append("  - Retakan pada x = %s m berdekatan dengan zona ini (sesuai)." % ", ".join(["%.0f" % v for v in dekat]))
    else:
        md.append("  - **Ketidaksesuaian**: tidak ada retakan tercatat di dekat zona ini (retakan di x = %s m)." % ", ".join(["%.0f" % v for v in x_retakan]))
md.append("")

md.append("## 5. Alternatif interpretasi\n")
alt = {"rendah": "resistivitas rendah dapat berasal dari lempung (kering sekalipun) atau dari tanah jenuh air; ERT saja tidak dapat membedakannya. Perlu data bor, uji kadar air, atau survei tambahan.",
       "menengah": "dapat berupa tanah lapuk, pasir lempungan, atau campuran lempung dan batuan; nilai ini ambigu.",
       "tinggi": "dapat berupa batuan kompak, tetapi juga material kering/lepas atau rongga; bukan otomatis batuan dasar yang stabil."}
penting = [r for r in zona if r["kelas"] in nama_kelas and (r["zone_id"] in kandidat or r["keyakinan"] in ["tinggi", "sedang"])]
if len(penting) == 0:
    penting = [r for r in zona if r["kelas"] in nama_kelas][:3]
for r in penting:
    md.append("- %s (%s): %s" % (r["zone_id"], r["kelas"], alt[r["kelas"]]))
md.append("")

md.append("## 6. Ringkasan untuk pengambil keputusan\n")
if len(kandidat) > 0:
    md.append("Pada penampang terlihat lapisan yang sangat basah/lunak (resistivitas rendah) di bawah lapisan yang lebih keras di dekat permukaan. "
              "Lapisan ini **diduga** dapat menjadi bidang tempat tanah bergerak. Ini baru dugaan dari satu lintasan, belum bukti, dan belum menyatakan seberapa berbahaya lereng ini.")
else:
    md.append("Pada penampang tidak terlihat lapisan basah/lunak yang jelas di bawah lapisan keras. Ini tidak berarti lereng aman; hanya satu lintasan yang diukur.")
md.append("")
md.append("## 7. Saran survei lanjutan\n")
md.append("- Bor tangan atau bor inti pada x dekat zona dugaan untuk memastikan material dan kedalaman muka air.")
md.append("- Tambah lintasan ERT sejajar dan tegak lurus lereng agar zona dapat dilihat tiga dimensi.")
md.append("- Ulangi pengukuran pada musim berbeda (time-lapse) dan pasang pengukur muka air/kadar air tanah.")
md.append("- Pemetaan retakan dan pengukuran gerakan tanah (inklinometer atau patok) untuk kuantifikasi.\n")
md.append("## 8. Keterbatasan\n")
md.append("- Inversi tidak dijalankan ulang; kualitas model mengikuti hasil inversi yang diberikan (RMS %.2f %%)." % rms_rata)
md.append("- Penampang hanya direncanakan dalam teks, belum digambar.")
md.append("- Tidak ada pernyataan tingkat bahaya longsor dalam laporan ini.\n")

# ----------------------------- VERIFIKASI ------------------------------------
md.append("## 9. Verifikasi\n")
ulang = pd.read_csv(PATH_ERT)
ulang["luar"] = luar_doi
for r in zona[:2]:
    zid = r["zone_id"]
    if zid not in zona_sel:
        continue
    gabung = zona_sel[zid].merge(ulang, on=["x_m", "z_m"], how="left")
    gabung = gabung[~gabung["luar"]]
    lo = round(float(gabung["rho_ohm_m"].min()), 1); hi = round(float(gabung["rho_ohm_m"].max()), 1)
    cocok = (lo == r["rho_min"]) and (hi == r["rho_max"])
    md.append("- Hitung ulang rentang rho %s dari `ert_inverted.csv`: %.1f - %.1f. Sama dengan zones.csv: **%s**." % (zid, lo, hi, "YA" if cocok else "TIDAK"))
    if not cocok:
        md.append("  - Perlu koreksi tabel zona.")
if len(banding) > 0 and len(x_retakan) > 0:
    for zid, xa, xb, dekat in banding:
        md.append("- Posisi %s vs retakan: %s." % (zid, "berdekatan" if len(dekat) > 0 else "TIDAK sesuai (dicatat sebagai ketidaksesuaian)"))
f = open(os.path.join(OUT_DIR, "interpretation.md"), "w", encoding="utf-8"); f.write("\n".join(md)); f.close()

# ----------------------------- CEK SHA-256 SESUDAH ---------------------------
for p_ in sha_sebelum:
    f = open(p_, "rb"); sesudah = hashlib.sha256(f.read()).hexdigest(); f.close()
    if sesudah != sha_sebelum[p_]:
        sys.exit("GAGAL: %s berubah selama proses!" % p_)

# ----------------------------- RUN CONFIG ------------------------------------
cfg = {"batas_kelas_ohm_m": {"rendah": "< %.0f" % RHO_RENDAH_MAX, "menengah": "%.0f - %.0f" % (RHO_RENDAH_MAX, RHO_TINGGI_MIN), "tinggi": ">= %.0f" % RHO_TINGGI_MIN},
       "alasan_batas": "Asumsi berdasarkan rentang umum: tanah lempung/jenuh air < ~100 ohm.m, tanah lapuk/pasir lempungan ~100-500 ohm.m, batuan kompak/material kering >= ~500 ohm.m. Rentang tumpang tindih; sesuaikan dengan data bor lokal.",
       "min_sel_zona": MIN_CELL, "min_kedalaman_bidang_gelincir_m": MIN_DEPTH_SLIP, "min_lebar_frac": MIN_LEBAR_FRAC,
       "toleransi_topografi_m": TOL_TOPO_M, "doi_flag_tersedia": ada_doi,
       "aturan_keyakinan": "tinggi: >=60 sel dan RMS<=%.0f%%; sedang: >=30 sel dan RMS<=%.0f%%; selain itu rendah" % (RMS_BAIK, RMS_BATAS),
       "inversi_dijalankan_ulang": False,
       "sha256_input": sha_sebelum,
       "versi": {"python": platform.python_version(), "numpy": np.__version__, "scipy": scipy.__version__, "pandas": pd.__version__}}
f = open(os.path.join(OUT_DIR, "run_config.json"), "w"); json.dump(cfg, f, indent=2); f.close()

# ----------------------------- HASIL AKHIR -----------------------------------
print("ERT selesai: zona dan interpretasi disimpan di", os.path.abspath(OUT_DIR))

ERT selesai: zona dan interpretasi disimpan di c:\Users\X270\Downloads\results\ert


## **Prompt 3 — AWS: QA/QC Data Automatic Weather Station 10 Menitan**

### **[R] Role**
Bertindaklah sebagai reviewer QC data meteorologi. Fokus pada preservasi data mentah, timestamp, satuan, dan transparansi ambang QC.

### **[T] Task**
Periksa data AWS `data/raw/aws_10min.csv` untuk nilai di luar rentang fisik, lonjakan tidak wajar, data beku (flatline), timestamp duplikat, dan data hilang, lalu tulis laporan QC dengan kolom flag terpisah. Jangan mengisi data hilang.

### **[C] Context**
Data AWS di `{{lokasi}}` periode `{{tanggal awal–akhir}}` akan dipakai untuk analisis iklim lokal dan validasi model cuaca. Laporan QC dipakai untuk memutuskan data mana yang layak dianalisis. Hasil tidak dipakai untuk analisis tren atau interpretasi iklim.

### **[I] Input**
- `data/raw/aws_10min.csv`: `timestamp` (zona waktu `{{WIB/UTC}}`), `temp_c` (°C), `rh_pct` (%), `pres_hpa` (hPa), `ws_ms` (m/s), `wd_deg` (derajat dari utara), `rad_wm2` (W/m²), `rain_mm` (mm per 10 menit).
- `data/raw/station_meta.json`: `lat`, `lon`, `elev_m`, tinggi sensor `{{m}}`.
- Ambang QC: `{{suhu min/max, RH 0–100, tekanan min/max, lonjakan maksimum per 10 menit, durasi flatline}}`.

### **[K] Constraints**
- Jangan mengubah `data/raw/`; jangan menghapus baris; flag disimpan pada kolom terpisah.
- Jangan mengisi data hilang (tanpa interpolasi atau imputasi).
- Setiap ambang ditulis eksplisit beserta alasannya; jangan menetapkan ambang tanpa menyebutnya.
- Jika zona waktu tidak jelas, jangan menebak: berhenti dan tanyakan.
- Software: Python dengan pandas dan NumPy; tanpa jaringan.

### **[O] Output (rencana teks)**
- `results/aws_qc/aws_10min_flagged.csv`: semua kolom asli ditambah `flag_range`, `flag_spike`, `flag_flatline`, `flag_missing`, `flag_duplicate`.
- `results/aws_qc/qc_report.md`: jumlah data, persentase data hilang per parameter, jumlah dan contoh waktu tiap jenis anomali, ambang beserta alasan, rekomendasi data layak/perlu dicek/dibuang.
- `results/aws_qc/summary.json`: `n_rows`, `n_flagged` per jenis, `pct_missing` per parameter.
- `results/aws_qc/run_config.json`: ambang dan versi package.

### **[Q] Quality criteria**
- Jumlah baris output = jumlah baris input.
- Persentase dihitung dari total baris input yang benar.
- Setiap flag memuat parameter, nilai, ambang, dan alasan.
- Metode dapat direproduksi dari `run_config.json`.
- Data mentah tidak berubah.

### **[V] Optional verification**
- Check: hitung SHA-256 `aws_10min.csv` sebelum dan sesudah. Expected: identik. Jika gagal: hentikan dan laporkan.
- Check: periksa manual 20 baris berflag acak. Expected: flag sesuai aturan. Jika ada yang salah: laporkan aturan yang bermasalah.
- Check: bandingkan sebagian hasil dengan stasiun terdekat `{{nama stasiun}}` bila ada. Expected: pola serupa. Jika tidak: catat sebagai temuan, bukan koreksi.



In [9]:
import os, sys, json, hashlib, platform
import numpy as np
import pandas as pd

# ----------------------------- PARAMETER (isi sesuai prompt) -----------------
LOKASI      = "Makassar, Sulawesi Selatan (data simulasi)"
PERIODE     = "1-10 Maret 2025"
ZONA_WAKTU  = "WITA"                 # WAJIB diisi: WIB / WITA / WIT / UTC. Bila kosong -> berhenti
STASIUN_PEMBANDING = "tidak ada data pembanding"
BUAT_DATA_SINTETIS = True            # True: bila aws_10min.csv belum ada, dibuat data SIMULASI

PARAM = ["temp_c", "rh_pct", "pres_hpa", "ws_ms", "wd_deg", "rad_wm2", "rain_mm"]
SATUAN = {"temp_c": "C", "rh_pct": "%", "pres_hpa": "hPa", "ws_ms": "m/s", "wd_deg": "derajat", "rad_wm2": "W/m2", "rain_mm": "mm/10 menit"}

# Ambang rentang fisik (min, max) beserta alasan
RENTANG = {"temp_c": (15.0, 40.0), "rh_pct": (0.0, 100.0), "pres_hpa": (None, None),     # tekanan dihitung dari elevasi stasiun
           "ws_ms": (0.0, 40.0), "wd_deg": (0.0, 360.0), "rad_wm2": (0.0, 1400.0), "rain_mm": (0.0, 50.0)}
ALASAN_RENTANG = {"temp_c": "suhu udara tropis dekat permukaan jarang di luar 15-40 C",
                  "rh_pct": "kelembapan relatif secara fisik 0-100 %",
                  "pres_hpa": "tekanan stasiun = tekanan laut standar yang dikoreksi elevasi, toleransi +/- 30 hPa",
                  "ws_ms": "kecepatan angin tidak negatif; > 40 m/s tidak realistis untuk sensor darat tropis",
                  "wd_deg": "arah angin 0-360 derajat dari utara",
                  "rad_wm2": "radiasi global tidak negatif; > 1400 W/m2 melebihi konstanta surya + pantulan awan",
                  "rain_mm": "hujan tidak negatif; > 50 mm per 10 menit sangat ekstrem untuk sensor tipping bucket"}
TOL_TEKANAN = 30.0

# Ambang lonjakan (selisih mutlak antar 2 data berurutan dengan jarak 10 menit)
LONJAKAN = {"temp_c": 5.0, "rh_pct": 30.0, "pres_hpa": 3.0, "ws_ms": 15.0, "rad_wm2": 800.0}
ALASAN_LONJAKAN = {"temp_c": "perubahan suhu > 5 C dalam 10 menit tidak wajar secara termal",
                   "rh_pct": "perubahan RH > 30 % dalam 10 menit biasanya kesalahan sensor (hujan deras dapat menyebabkan, perlu cek)",
                   "pres_hpa": "perubahan tekanan > 3 hPa dalam 10 menit tidak wajar kecuali badai ekstrem",
                   "ws_ms": "perubahan angin > 15 m/s dalam 10 menit tidak wajar",
                   "rad_wm2": "perubahan radiasi > 800 W/m2 dalam 10 menit tidak wajar (awan cepat lewat tidak sebesar ini)"}

# Ambang flatline: jumlah data berurutan identik (10 menit per data)
FLATLINE_N = {"temp_c": 6, "rh_pct": 6, "pres_hpa": 6, "ws_ms": 12, "wd_deg": 12, "rad_wm2": 6}
ALASAN_FLATLINE = {"temp_c": ">= 6 data (1 jam) identik: suhu hampir selalu berubah walau kecil",
                   "rh_pct": ">= 6 data (1 jam) identik: RH hampir selalu berubah",
                   "pres_hpa": ">= 6 data (1 jam) identik: tekanan hampir selalu berubah (resolusi 0.1 hPa)",
                   "ws_ms": ">= 12 data (2 jam) identik dan bukan 0: angin jarang konstan",
                   "wd_deg": ">= 12 data (2 jam) identik dan bukan 0: arah angin jarang konstan",
                   "rad_wm2": ">= 6 data identik dan bukan 0 (nilai 0 malam hari wajar)"}
ABAIKAN_NOL = ["ws_ms", "wd_deg", "rad_wm2"]     # nilai 0 pada parameter ini tidak dianggap flatline

# Rekomendasi per parameter berdasarkan persen baris bermasalah
BATAS_LAYAK = 5.0       # < 5 %  -> layak
BATAS_CEK   = 20.0      # 5-20 % -> perlu dicek; > 20 % -> dibuang (dari analisis)

PATH_DATA = "data/raw/aws_10min.csv"
PATH_META = "data/raw/station_meta.json"
OUT_DIR   = "results/aws_qc"

# ----------------------------- DATA SINTETIS (bila file belum ada) -----------
# PERHATIAN: ini data simulasi untuk uji skrip, BUKAN data lapangan.
if (not os.path.exists(PATH_DATA)) and BUAT_DATA_SINTETIS:
    os.makedirs(os.path.dirname(PATH_DATA), exist_ok=True)
    rng_s = np.random.default_rng(3)
    ts_s = pd.date_range("2025-03-01 00:00", periods=144 * 10, freq="10min")
    jam_s = ts_s.hour.values + ts_s.minute.values / 60.0
    d_s = pd.DataFrame({"timestamp": ts_s.strftime("%Y-%m-%d %H:%M:%S")})
    d_s["temp_c"] = np.round(26 + 5 * np.sin((jam_s - 9) / 24 * 2 * np.pi) + rng_s.normal(0, 0.15, len(ts_s)), 1)
    d_s["rh_pct"] = np.round(np.clip(80 - 15 * np.sin((jam_s - 9) / 24 * 2 * np.pi) + rng_s.normal(0, 1, len(ts_s)), 30, 100), 1)
    d_s["pres_hpa"] = np.round(1010 + 1.5 * np.sin(jam_s / 12 * np.pi) + rng_s.normal(0, 0.1, len(ts_s)), 1)
    d_s["ws_ms"] = np.round(np.abs(rng_s.normal(2, 1, len(ts_s))), 1)
    d_s["wd_deg"] = np.round(rng_s.uniform(0, 360, len(ts_s)), 0)
    d_s["rad_wm2"] = np.round(np.clip(900 * np.sin((jam_s - 6) / 12 * np.pi), 0, None) + np.where(np.sin((jam_s - 6) / 12 * np.pi) > 0, rng_s.normal(0, 5, len(ts_s)), 0), 1)
    d_s["rad_wm2"] = np.clip(d_s["rad_wm2"], 0, None)
    d_s["rain_mm"] = np.round(np.where(rng_s.random(len(ts_s)) < 0.02, rng_s.exponential(1.0, len(ts_s)), 0.0), 1)
    d_s.loc[50, "temp_c"] = 99.9                      # di luar rentang
    d_s.loc[200, "temp_c"] = d_s.loc[199, "temp_c"] + 9    # lonjakan
    d_s.loc[300:312, "rh_pct"] = 77.7                 # flatline
    d_s.loc[400, "pres_hpa"] = np.nan                 # hilang
    d_s.loc[401, "ws_ms"] = np.nan
    d_s.loc[600, "rain_mm"] = -1.0                    # hujan negatif
    d_s = pd.concat([d_s, d_s.iloc[[700]]], ignore_index=True)   # timestamp duplikat
    d_s.to_csv(PATH_DATA, index=False)
    f = open(PATH_META, "w"); json.dump({"lat": -5.1, "lon": 119.5, "elev_m": 20.0, "tinggi_sensor_m": 2.0}, f); f.close()

if not os.path.exists(PATH_DATA):
    sys.exit("File tidak ditemukan: %s\nFolder kerja saat ini: %s" % (PATH_DATA, os.getcwd()))
if ZONA_WAKTU not in ["WIB", "WITA", "WIT", "UTC"]:
    sys.exit("BERHENTI: zona waktu tidak jelas (ZONA_WAKTU = '%s'). Isi WIB/WITA/WIT/UTC, jangan menebak." % ZONA_WAKTU)

# ----------------------------- CEK SHA-256 SEBELUM ---------------------------
f = open(PATH_DATA, "rb"); sha_sebelum = hashlib.sha256(f.read()).hexdigest(); f.close()
os.makedirs(OUT_DIR, exist_ok=True)

# ----------------------------- BACA DATA -------------------------------------
df = pd.read_csv(PATH_DATA)
for k in ["timestamp"] + PARAM:
    if k not in df.columns:
        sys.exit("Kolom wajib tidak ada: " + k)
n_rows = len(df)
meta = {}
if os.path.exists(PATH_META):
    f = open(PATH_META, "r"); meta = json.load(f); f.close()
elev = float(meta.get("elev_m", 0.0))
p_ref = 1013.25 * (1 - 2.25577e-5 * elev) ** 5.25588         # tekanan standar pada elevasi stasiun
RENTANG["pres_hpa"] = (p_ref - TOL_TEKANAN, p_ref + TOL_TEKANAN)

df["_ts"] = pd.to_datetime(df["timestamp"], errors="coerce")
n_ts_rusak = int(df["_ts"].isna().sum())

# ----------------------------- FLAG -------------------------------------------
catatan = []        # setiap flag: jenis, baris, timestamp, parameter, nilai, ambang, alasan
f_range = [[] for _ in range(n_rows)]
f_spike = [[] for _ in range(n_rows)]
f_flat = [[] for _ in range(n_rows)]
f_miss = [[] for _ in range(n_rows)]
f_dup = [""] * n_rows

# -- data hilang & rentang fisik
for p in PARAM:
    v = df[p].values.astype(float)
    lo, hi = RENTANG[p]
    for i in range(n_rows):
        if np.isnan(v[i]):
            f_miss[i].append(p)
            catatan.append(("missing", i, df["timestamp"].iloc[i], p, "NaN", "tidak boleh kosong", "data hilang (tidak diisi)"))
        elif v[i] < lo or v[i] > hi:
            f_range[i].append(p)
            catatan.append(("range", i, df["timestamp"].iloc[i], p, v[i], "%.1f s/d %.1f %s" % (lo, hi, SATUAN[p]), ALASAN_RENTANG[p]))

# -- timestamp duplikat (semua baris yang berbagi timestamp ditandai)
dup = df["timestamp"].duplicated(keep=False).values
for i in range(n_rows):
    if dup[i]:
        f_dup[i] = "duplikat"
        catatan.append(("duplicate", i, df["timestamp"].iloc[i], "timestamp", df["timestamp"].iloc[i], "timestamp unik", "timestamp muncul lebih dari sekali"))

# -- lonjakan (hanya bila data sebelumnya tepat 10 menit sebelumnya)
ts = df["_ts"]
selang = ts.diff().dt.total_seconds().values
for p, batas in LONJAKAN.items():
    v = df[p].values.astype(float)
    for i in range(1, n_rows):
        if np.isnan(v[i]) or np.isnan(v[i - 1]) or np.isnan(selang[i]) or selang[i] != 600:
            continue
        if abs(v[i] - v[i - 1]) > batas:
            f_spike[i].append(p)
            catatan.append(("spike", i, df["timestamp"].iloc[i], p, v[i], "selisih mutlak <= %.1f %s (ditemukan %.2f)" % (batas, SATUAN[p], abs(v[i] - v[i - 1])), ALASAN_LONJAKAN[p]))

# -- flatline (nilai identik berurutan, NaN memutus run)
for p, nmin in FLATLINE_N.items():
    v = df[p].values.astype(float)
    i = 0
    while i < n_rows:
        if np.isnan(v[i]):
            i += 1; continue
        j = i
        while j + 1 < n_rows and (not np.isnan(v[j + 1])) and v[j + 1] == v[i]:
            j += 1
        panjang = j - i + 1
        if panjang >= nmin and not (p in ABAIKAN_NOL and v[i] == 0):
            for r in range(i, j + 1):
                f_flat[r].append(p)
            catatan.append(("flatline", i, df["timestamp"].iloc[i], p, v[i], ">= %d data identik berurutan (ditemukan %d, s/d %s)" % (nmin, panjang, df["timestamp"].iloc[j]), ALASAN_FLATLINE[p]))
        i = j + 1

# -- tulis kolom flag (kolom asli tidak diubah)
out = pd.read_csv(PATH_DATA)
out["flag_range"] = [";".join(x) if len(x) > 0 else "ok" for x in f_range]
out["flag_spike"] = [";".join(x) if len(x) > 0 else "ok" for x in f_spike]
out["flag_flatline"] = [";".join(x) if len(x) > 0 else "ok" for x in f_flat]
out["flag_missing"] = [";".join(x) if len(x) > 0 else "ok" for x in f_miss]
out["flag_duplicate"] = [x if x != "" else "ok" for x in f_dup]
out.to_csv(os.path.join(OUT_DIR, "aws_10min_flagged.csv"), index=False)
if len(out) != n_rows:
    sys.exit("GAGAL: jumlah baris output (%d) tidak sama dengan input (%d)" % (len(out), n_rows))

# ----------------------------- RINGKASAN -------------------------------------
kolom_flag = ["flag_range", "flag_spike", "flag_flatline", "flag_missing", "flag_duplicate"]
n_flagged = {k: int((out[k] != "ok").sum()) for k in kolom_flag}
pct_missing = {p: round(float(df[p].isna().mean() * 100.0), 3) for p in PARAM}

# baris bermasalah per parameter (untuk rekomendasi)
rekom = {}
for p in PARAM:
    bad = np.zeros(n_rows, dtype=bool)
    for lst in [f_range, f_spike, f_flat, f_miss]:
        bad = bad | np.array([p in x for x in lst])
    pct = bad.mean() * 100.0
    rekom[p] = (int(bad.sum()), pct, "layak" if pct < BATAS_LAYAK else ("perlu dicek" if pct <= BATAS_CEK else "dibuang dari analisis"))

# celah waktu (timestamp hilang, baris tidak ditambah)
ts_ok = ts.dropna().drop_duplicates().sort_values()
if len(ts_ok) > 1:
    rentang_penuh = pd.date_range(ts_ok.iloc[0], ts_ok.iloc[-1], freq="10min")
    n_celah = len(rentang_penuh.difference(ts_ok))
else:
    n_celah = 0

summary = {"n_rows": n_rows, "n_flagged": n_flagged, "pct_missing": pct_missing}
f = open(os.path.join(OUT_DIR, "summary.json"), "w"); json.dump(summary, f, indent=2); f.close()

# ----------------------------- VERIFIKASI 20 BARIS BERFLAG -------------------
rng = np.random.default_rng(42)
pilih = rng.choice(len(catatan), size=min(20, len(catatan)), replace=False) if len(catatan) > 0 else []
salah = []
for q in pilih:
    jenis, i, tstamp, p, nilai, ambang, alasan = catatan[q]
    ok = True
    if jenis == "missing":
        ok = bool(pd.isna(df[p].iloc[i]))
    elif jenis == "range":
        x = float(df[p].iloc[i]); ok = (x < RENTANG[p][0]) or (x > RENTANG[p][1])
    elif jenis == "duplicate":
        ok = int((df["timestamp"] == tstamp).sum()) > 1
    elif jenis == "spike":
        ok = abs(float(df[p].iloc[i]) - float(df[p].iloc[i - 1])) > LONJAKAN[p]
    elif jenis == "flatline":
        a = i; b = i
        while a - 1 >= 0 and df[p].iloc[a - 1] == df[p].iloc[i]: a -= 1
        while b + 1 < n_rows and df[p].iloc[b + 1] == df[p].iloc[i]: b += 1
        ok = (b - a + 1) >= FLATLINE_N[p]
    if not ok:
        salah.append((jenis, i, p))

# ----------------------------- LAPORAN QC (Markdown) -------------------------
md = []
md.append("# Laporan QA/QC AWS 10 Menitan\n")
md.append("Lokasi: %s | Periode: %s | Zona waktu data: **%s**\n" % (LOKASI, PERIODE, ZONA_WAKTU))
md.append("> Data mentah tidak diubah, tidak ada baris dihapus, data hilang tidak diisi. Flag disimpan pada kolom terpisah.\n")
md.append("## 1. Ringkasan\n")
md.append("- Jumlah baris: **%d** (output = input)." % n_rows)
md.append("- Rentang waktu: %s sampai %s." % (ts_ok.iloc[0], ts_ok.iloc[-1]) if len(ts_ok) > 0 else "- Timestamp tidak dapat dibaca.")
md.append("- Timestamp tidak terbaca: %d; celah waktu 10 menit (timestamp hilang, tidak ditambahkan): %d." % (n_ts_rusak, n_celah))
md.append("- Elevasi stasiun: %.1f m -> tekanan acuan %.1f hPa (rentang diterima %.1f - %.1f hPa)." % (elev, p_ref, RENTANG["pres_hpa"][0], RENTANG["pres_hpa"][1]))
md.append("")
md.append("## 2. Persentase data hilang per parameter (dari total %d baris)\n" % n_rows)
md.append("| Parameter | Satuan | Hilang (baris) | Hilang (%) |")
md.append("|---|---|---|---|")
for p in PARAM:
    md.append("| %s | %s | %d | %.3f |" % (p, SATUAN[p], int(df[p].isna().sum()), pct_missing[p]))
md.append("")
md.append("## 3. Jumlah baris berflag per jenis\n")
md.append("| Flag | Jumlah baris | Persen dari total |")
md.append("|---|---|---|")
for k in kolom_flag:
    md.append("| %s | %d | %.3f |" % (k, n_flagged[k], n_flagged[k] / n_rows * 100.0))
md.append("")
md.append("## 4. Contoh waktu tiap jenis anomali (maks. 5 contoh)\n")
for jenis in ["range", "spike", "flatline", "missing", "duplicate"]:
    isi = [c for c in catatan if c[0] == jenis]
    md.append("### %s (%d kejadian)" % (jenis, len(isi)))
    if len(isi) == 0:
        md.append("Tidak ada.\n"); continue
    md.append("| Timestamp | Parameter | Nilai | Ambang | Alasan |")
    md.append("|---|---|---|---|---|")
    for c in isi[:5]:
        md.append("| %s | %s | %s | %s | %s |" % (c[2], c[3], c[4], c[5], c[6]))
    md.append("")
md.append("## 5. Ambang QC dan alasan\n")
md.append("| Parameter | Rentang fisik | Alasan rentang | Lonjakan maks / 10 menit | Flatline (jumlah data) |")
md.append("|---|---|---|---|---|")
for p in PARAM:
    lo, hi = RENTANG[p]
    md.append("| %s | %.1f - %.1f %s | %s | %s | %s |" % (p, lo, hi, SATUAN[p], ALASAN_RENTANG[p],
              ("%.1f (%s)" % (LONJAKAN[p], ALASAN_LONJAKAN[p])) if p in LONJAKAN else "tidak dipakai",
              ("%d (%s)" % (FLATLINE_N[p], ALASAN_FLATLINE[p])) if p in FLATLINE_N else "tidak dipakai"))
md.append("")
md.append("Ambang ini asumsi awal untuk iklim tropis dan harus dikonfirmasi; ambang diatur di bagian PARAMETER skrip dan dicatat di `run_config.json`.\n")
md.append("## 6. Rekomendasi data (per parameter)\n")
md.append("Aturan: < %.0f %% baris bermasalah = layak; %.0f - %.0f %% = perlu dicek; > %.0f %% = dibuang dari analisis (hanya rekomendasi, data tidak dihapus).\n" % (BATAS_LAYAK, BATAS_LAYAK, BATAS_CEK, BATAS_CEK))
md.append("| Parameter | Baris bermasalah | Persen | Rekomendasi |")
md.append("|---|---|---|---|")
for p in PARAM:
    md.append("| %s | %d | %.3f | %s |" % (p, rekom[p][0], rekom[p][1], rekom[p][2]))
md.append("")
md.append("## 7. Verifikasi\n")
md.append("- SHA-256 `aws_10min.csv` sebelum dan sesudah: lihat bagian akhir (identik bila skrip selesai tanpa error).")
md.append("- Pemeriksaan ulang %d catatan flag acak: %s." % (len(pilih), "semua sesuai aturan" if len(salah) == 0 else "ada yang tidak sesuai -> " + str(salah)))
md.append("- Perbandingan dengan stasiun terdekat (%s): belum dilakukan (tidak ada data pembanding); catat sebagai temuan, bukan koreksi.\n" % STASIUN_PEMBANDING)
f = open(os.path.join(OUT_DIR, "qc_report.md"), "w", encoding="utf-8"); f.write("\n".join(md)); f.close()

# ----------------------------- CEK SHA-256 SESUDAH ---------------------------
f = open(PATH_DATA, "rb"); sha_sesudah = hashlib.sha256(f.read()).hexdigest(); f.close()
if sha_sebelum != sha_sesudah:
    sys.exit("GAGAL: aws_10min.csv berubah selama proses!")

# ----------------------------- RUN CONFIG ------------------------------------
cfg = {"zona_waktu": ZONA_WAKTU,
       "rentang_fisik": {p: [RENTANG[p][0], RENTANG[p][1]] for p in PARAM}, "alasan_rentang": ALASAN_RENTANG,
       "toleransi_tekanan_hpa": TOL_TEKANAN, "elevasi_stasiun_m": elev, "tekanan_acuan_hpa": round(p_ref, 2),
       "lonjakan_per_10menit": LONJAKAN, "alasan_lonjakan": ALASAN_LONJAKAN,
       "flatline_n_data": FLATLINE_N, "alasan_flatline": ALASAN_FLATLINE, "flatline_abaikan_nol": ABAIKAN_NOL,
       "batas_rekomendasi_pct": {"layak_di_bawah": BATAS_LAYAK, "perlu_dicek_hingga": BATAS_CEK},
       "sha256_aws_10min": sha_sesudah, "sha256_identik": sha_sebelum == sha_sesudah,
       "versi": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__}}
f = open(os.path.join(OUT_DIR, "run_config.json"), "w"); json.dump(cfg, f, indent=2); f.close()

# ----------------------------- HASIL AKHIR -----------------------------------
print("QC AWS selesai: data berflag dan laporan QC disimpan di", os.path.abspath(OUT_DIR))

QC AWS selesai: data berflag dan laporan QC disimpan di c:\Users\X270\Downloads\results\aws_qc


## **Prompt 4 — AWS: Pola Diurnal dan Variasi Bulanan Suhu dan Curah Hujan**

### **[R] Role**
Bertindaklah sebagai reviewer analisis klimatologi deskriptif. Fokus pada ukuran sampel, statistik yang sesuai, dan batas kesimpulan yang boleh ditarik dari data.

### **[T] Task**
Hitung siklus diurnal suhu dan curah hujan serta variasi bulanan dari data AWS yang sudah lolos QC `data/processed/aws_clean.csv`, lalu jelaskan hasilnya secara fisis. Jangan menyimpulkan tren jangka panjang.

### **[C] Context**
Lokasi di wilayah tropis `{{pesisir/dataran tinggi}}` dengan pengaruh monsun. Hasil dipakai untuk bab pembahasan tugas akhir. Hasil bukan analisis perubahan iklim dan bukan prakiraan cuaca.

### **[I] Input**
- `data/processed/aws_clean.csv`: `timestamp` (`{{WIB/UTC}}`), `temp_c` (°C), `rain_mm` (mm per 10 menit), `qc_ok` (boolean). Periode `{{minimal 1 tahun}}`.
- Definisi musim: `{{mis. hujan Nov–Mar, kemarau Mei–Sep}}`, sumber `{{BMKG/literatur}}`.
- Ambang hujan ekstrem: `{{mm/hari}}`.

### **[K] Constraints**
- Jangan mengubah `data/processed/`; baris dengan `qc_ok = False` dikeluarkan dan jumlahnya dilaporkan.
- Gunakan rata-rata, median, dan persentil; jangan hanya rata-rata.
- Jangan menarik kesimpulan tren jangka panjang dari data kurang dari 10 tahun; tulis keterbatasannya.
- Bedakan korelasi dan sebab-akibat.
- Software: Python dengan pandas dan NumPy; tanpa jaringan.

### **[O] Output (rencana teks)**
- `results/aws_diurnal/diurnal_stats.csv`: `jam`, `temp_mean`, `temp_median`, `temp_p25`, `temp_p75`, `rain_mean_mm`, `n` (per jam lokal).
- `results/aws_diurnal/monthly_stats.csv`: `bulan`, `temp_mean`, `rain_total_mm`, `n_hari_valid`, `n_hari_hujan`.
- `results/aws_diurnal/analysis.md`: jam puncak suhu dan hujan, perbandingan musim hujan dan kemarau, hari hujan ekstrem, interpretasi fisis (pemanasan siang, konveksi, monsun), keterbatasan. Grafik hanya direncanakan (mis. boxplot per jam), belum digambar.
- `results/aws_diurnal/run_config.json`: zona waktu, definisi musim, ambang ekstrem.

### **[Q] Quality criteria**
- Setiap angka disertai satuan dan jumlah sampel `n`.
- Jumlah data yang dikeluarkan karena QC dilaporkan.
- Interpretasi fisis sesuai teori dasar meteorologi dan tidak melampaui data.
- Zona waktu tercatat dan konsisten.

### **[V] Optional verification**
- Check: hitung ulang median suhu pada 3 jam berbeda secara independen. Expected: sama dengan `diurnal_stats.csv`. Jika gagal: laporkan perbedaan.
- Check: bandingkan pola bulanan dengan normal klimatologi `{{BMKG wilayah terdekat}}`. Expected: pola musim serupa. Jika tidak: catat dan diskusikan kemungkinan penyebab.

In [10]:
import os, sys, json, hashlib, platform
import numpy as np
import pandas as pd

# ----------------------------- PARAMETER (isi sesuai prompt) -----------------
LOKASI        = "Makassar, Sulawesi Selatan (data simulasi)"
TIPE_WILAYAH  = "pesisir"
ZONA_DATA     = "WITA"                         # zona waktu timestamp di file: WIB/WITA/WIT/UTC
ZONA_LOKAL    = "WITA"                         # zona waktu lokal untuk perhitungan jam
OFFSET_JAM    = {"UTC": 0, "WIB": 7, "WITA": 8, "WIT": 9}
MUSIM_HUJAN   = [11, 12, 1, 2, 3]              # contoh: Nov-Mar     (sumber: {{BMKG/literatur}})
MUSIM_KEMARAU = [5, 6, 7, 8, 9]                # contoh: Mei-Sep
SUMBER_MUSIM  = "BMKG"
HUJAN_EKSTREM_MM_HARI = 50.0                   # ambang hari hujan lebat/ekstrem (mm/hari), {{isi sesuai prompt}}
HARI_HUJAN_MM = 1.0                            # hari hujan bila total >= 1 mm/hari
MIN_FRAC_HARI = 0.8                            # hari valid bila >= 80 % dari 144 data 10 menit valid
MIN_DATA_JAM_HUJAN = 6                         # jam dihitung bila 6 data 10 menit (1 jam penuh) valid
BUAT_DATA_SINTETIS = True                      # True: bila aws_clean.csv belum ada, dibuat data SIMULASI

PATH_DATA = "data/processed/aws_clean.csv"
OUT_DIR   = "results/aws_diurnal"

# ----------------------------- DATA SINTETIS (bila file belum ada) -----------
# PERHATIAN: ini data simulasi untuk uji skrip, BUKAN data lapangan.
if (not os.path.exists(PATH_DATA)) and BUAT_DATA_SINTETIS:
    os.makedirs(os.path.dirname(PATH_DATA), exist_ok=True)
    rng_s = np.random.default_rng(4)
    ts_s = pd.date_range("2025-01-01 00:00", "2025-12-31 23:50", freq="10min")
    jam_s = ts_s.hour.values + ts_s.minute.values / 60.0
    bln_s = ts_s.month.values
    temp_s = 27 + 4.5 * np.sin((jam_s - 9) / 24 * 2 * np.pi) + 0.8 * np.cos((bln_s - 10) / 12 * 2 * np.pi) + rng_s.normal(0, 0.4, len(ts_s))
    p_hujan_s = np.where(np.isin(bln_s, MUSIM_HUJAN), 0.045, 0.008) * (1 + 1.5 * np.exp(-((jam_s - 15.5) ** 2) / 8.0))
    rain_s = np.where(rng_s.random(len(ts_s)) < p_hujan_s, rng_s.exponential(1.8, len(ts_s)), 0.0)
    qc_s = rng_s.random(len(ts_s)) > 0.02
    pd.DataFrame({"timestamp": ts_s.strftime("%Y-%m-%d %H:%M:%S"), "temp_c": np.round(temp_s, 1),
                  "rain_mm": np.round(rain_s, 1), "qc_ok": qc_s}).to_csv(PATH_DATA, index=False)

if not os.path.exists(PATH_DATA):
    sys.exit("File tidak ditemukan: %s\nFolder kerja saat ini: %s" % (PATH_DATA, os.getcwd()))
for z_ in [ZONA_DATA, ZONA_LOKAL]:
    if z_ not in OFFSET_JAM:
        sys.exit("BERHENTI: zona waktu '%s' tidak dikenal. Isi WIB/WITA/WIT/UTC." % z_)

# ----------------------------- CEK SHA-256 SEBELUM ---------------------------
f = open(PATH_DATA, "rb"); sha_sebelum = hashlib.sha256(f.read()).hexdigest(); f.close()
os.makedirs(OUT_DIR, exist_ok=True)

# ----------------------------- BACA DATA & QC --------------------------------
df = pd.read_csv(PATH_DATA)
for k in ["timestamp", "temp_c", "rain_mm", "qc_ok"]:
    if k not in df.columns:
        sys.exit("Kolom wajib tidak ada: " + k)
n_total = len(df)
df["qc_ok"] = df["qc_ok"].astype(str).str.strip().str.lower().isin(["true", "1", "1.0", "yes", "ya"])
n_keluar = int((~df["qc_ok"]).sum())
d = df[df["qc_ok"]].copy()                       # baris qc_ok = False dikeluarkan
d["ts"] = pd.to_datetime(d["timestamp"], errors="coerce")
n_ts_rusak = int(d["ts"].isna().sum())
d = d[d["ts"].notna()].copy()

# konversi ke zona waktu lokal (zona tercatat)
d["ts_lokal"] = d["ts"] + pd.Timedelta(hours=OFFSET_JAM[ZONA_LOKAL] - OFFSET_JAM[ZONA_DATA])
d["tanggal"] = d["ts_lokal"].dt.floor("D")
d["jam"] = d["ts_lokal"].dt.hour
d["bulan"] = d["ts_lokal"].dt.to_period("M").astype(str)
d["bln_no"] = d["ts_lokal"].dt.month
d["temp_ok"] = d["temp_c"].notna()
d["rain_ok"] = d["rain_mm"].notna()

periode_hari = (d["ts_lokal"].max() - d["ts_lokal"].min()).days + 1
periode_tahun = periode_hari / 365.25

# ----------------------------- STATISTIK DIURNAL -----------------------------
# hujan per jam: jumlah hujan per (tanggal, jam) dengan >= 6 data 10 menit valid
hj = d[d["rain_ok"]].groupby(["tanggal", "jam"]).agg(rain_jam=("rain_mm", "sum"), n10=("rain_mm", "count")).reset_index()
hj = hj[hj["n10"] >= MIN_DATA_JAM_HUJAN]

baris = []
for j in range(24):
    t = d.loc[(d["jam"] == j) & d["temp_ok"], "temp_c"].values
    r = hj.loc[hj["jam"] == j, "rain_jam"].values
    baris.append({"jam": j,
                  "temp_mean": round(float(np.mean(t)), 2) if len(t) > 0 else np.nan,
                  "temp_median": round(float(np.median(t)), 2) if len(t) > 0 else np.nan,
                  "temp_p25": round(float(np.percentile(t, 25)), 2) if len(t) > 0 else np.nan,
                  "temp_p75": round(float(np.percentile(t, 75)), 2) if len(t) > 0 else np.nan,
                  "rain_mean_mm": round(float(np.mean(r)), 3) if len(r) > 0 else np.nan,
                  "n": int(len(t)),
                  "n_jam_hujan": int(len(r))})
df_diu = pd.DataFrame(baris)
df_diu.to_csv(os.path.join(OUT_DIR, "diurnal_stats.csv"), index=False)

# ----------------------------- STATISTIK HARIAN & BULANAN --------------------
hari = d.groupby("tanggal").agg(n_temp=("temp_ok", "sum"), n_rain=("rain_ok", "sum"),
                                temp_mean=("temp_c", "mean"), rain_hari=("rain_mm", "sum")).reset_index()
min_n = int(MIN_FRAC_HARI * 144)
hari["valid"] = (hari["n_temp"] >= min_n) & (hari["n_rain"] >= min_n)
hari["bulan"] = hari["tanggal"].dt.to_period("M").astype(str)
hari["bln_no"] = hari["tanggal"].dt.month
hv = hari[hari["valid"]]

baris = []
for b in sorted(hari["bulan"].unique()):
    sub = hv[hv["bulan"] == b]
    baris.append({"bulan": b,
                  "temp_mean": round(float(sub["temp_mean"].mean()), 2) if len(sub) > 0 else np.nan,
                  "rain_total_mm": round(float(sub["rain_hari"].sum()), 1) if len(sub) > 0 else np.nan,
                  "n_hari_valid": int(len(sub)),
                  "n_hari_hujan": int((sub["rain_hari"] >= HARI_HUJAN_MM).sum())})
df_bln = pd.DataFrame(baris)
df_bln.to_csv(os.path.join(OUT_DIR, "monthly_stats.csv"), index=False)

# ----------------------------- MUSIM -----------------------------------------
def_musim = []
for m_ in hv["bln_no"]:
    def_musim.append("hujan" if m_ in MUSIM_HUJAN else ("kemarau" if m_ in MUSIM_KEMARAU else "peralihan"))
hv = hv.assign(musim=def_musim)
tabel_musim = []
for ms in ["hujan", "kemarau", "peralihan"]:
    sub = hv[hv["musim"] == ms]
    if len(sub) == 0:
        continue
    tabel_musim.append((ms, len(sub), sub["temp_mean"].mean(), sub["temp_mean"].median(),
                        sub["rain_hari"].mean(), sub["rain_hari"].median(), np.percentile(sub["rain_hari"], 90),
                        int((sub["rain_hari"] >= HARI_HUJAN_MM).sum())))

# ----------------------------- HARI HUJAN EKSTREM ----------------------------
ekstrem = hv[hv["rain_hari"] >= HUJAN_EKSTREM_MM_HARI].sort_values("rain_hari", ascending=False)

# ----------------------------- JAM PUNCAK ------------------------------------
dd = df_diu.dropna(subset=["temp_median"])
jam_puncak_suhu = int(dd.loc[dd["temp_median"].idxmax(), "jam"])
jam_min_suhu = int(dd.loc[dd["temp_median"].idxmin(), "jam"])
dr = df_diu.dropna(subset=["rain_mean_mm"])
jam_puncak_hujan = int(dr.loc[dr["rain_mean_mm"].idxmax(), "jam"]) if len(dr) > 0 else None

# ----------------------------- VERIFIKASI (hitung ulang independen) ----------
jam_cek = [jam_min_suhu, jam_puncak_suhu, 12]
ulang = pd.read_csv(PATH_DATA)
ulang = ulang[ulang["qc_ok"].astype(str).str.strip().str.lower().isin(["true", "1", "1.0", "yes", "ya"])]
ulang["jam_l"] = (pd.to_datetime(ulang["timestamp"]) + pd.Timedelta(hours=OFFSET_JAM[ZONA_LOKAL] - OFFSET_JAM[ZONA_DATA])).dt.hour
hasil_cek = []
for j in jam_cek:
    nilai = ulang.loc[(ulang["jam_l"] == j) & ulang["temp_c"].notna(), "temp_c"].values
    med = round(float(np.median(nilai)), 2)
    acuan = float(df_diu.loc[df_diu["jam"] == j, "temp_median"].iloc[0])
    hasil_cek.append((j, med, acuan, abs(med - acuan) < 1e-9))

# ----------------------------- ANALISIS (Markdown) ---------------------------
md = []
md.append("# Analisis Pola Diurnal dan Variasi Bulanan\n")
md.append("Lokasi: %s (%s) | Zona waktu data: %s | Zona waktu analisis: **%s**\n" % (LOKASI, TIPE_WILAYAH, ZONA_DATA, ZONA_LOKAL))
md.append("> Analisis deskriptif. Bukan analisis perubahan iklim dan bukan prakiraan cuaca.\n")
md.append("## 1. Data yang dipakai\n")
md.append("- Total baris input: %d; dikeluarkan karena `qc_ok = False`: **%d** (%.2f %%); timestamp tidak terbaca: %d." % (n_total, n_keluar, n_keluar / n_total * 100.0, n_ts_rusak))
md.append("- Baris dipakai: %d; periode: %s sampai %s (%.2f tahun)." % (len(d), d["ts_lokal"].min(), d["ts_lokal"].max(), periode_tahun))
md.append("- Hari valid (>= %.0f %% data valid): %d dari %d hari." % (MIN_FRAC_HARI * 100, int(hari["valid"].sum()), len(hari)))
if periode_hari < 365:
    md.append("- **Peringatan**: periode kurang dari 1 tahun; siklus musiman tidak lengkap.")
md.append("")
md.append("## 2. Siklus diurnal\n")
md.append("- Jam puncak suhu (median): **%02d:00** dengan median %.2f C (n = %d); jam terendah: **%02d:00** dengan median %.2f C (n = %d)." % (
    jam_puncak_suhu, df_diu.loc[df_diu["jam"] == jam_puncak_suhu, "temp_median"].iloc[0], df_diu.loc[df_diu["jam"] == jam_puncak_suhu, "n"].iloc[0],
    jam_min_suhu, df_diu.loc[df_diu["jam"] == jam_min_suhu, "temp_median"].iloc[0], df_diu.loc[df_diu["jam"] == jam_min_suhu, "n"].iloc[0]))
amp = df_diu.loc[df_diu["jam"] == jam_puncak_suhu, "temp_median"].iloc[0] - df_diu.loc[df_diu["jam"] == jam_min_suhu, "temp_median"].iloc[0]
md.append("- Amplitudo diurnal (median maks - median min): %.2f C." % amp)
if jam_puncak_hujan is not None:
    md.append("- Jam puncak hujan (rata-rata hujan per jam): **%02d:00** dengan %.3f mm/jam (n jam = %d)." % (
        jam_puncak_hujan, df_diu.loc[df_diu["jam"] == jam_puncak_hujan, "rain_mean_mm"].iloc[0], df_diu.loc[df_diu["jam"] == jam_puncak_hujan, "n_jam_hujan"].iloc[0]))
md.append("- Kuartil (P25-P75) suhu per jam ada di `diurnal_stats.csv`; rata-rata hujan per jam dipengaruhi kejadian jarang yang besar, sehingga baca bersama sebaran.")
md.append("- Grafik (boxplot suhu per jam, batang hujan per jam) hanya direncanakan, belum digambar.\n")
md.append("## 3. Variasi bulanan\n")
md.append("| Bulan | Suhu rata-rata (C) | Hujan total (mm) | Hari valid (n) | Hari hujan (>= %.1f mm) |" % HARI_HUJAN_MM)
md.append("|---|---|---|---|---|")
for _, r in df_bln.iterrows():
    md.append("| %s | %s | %s | %d | %d |" % (r["bulan"], "-" if pd.isna(r["temp_mean"]) else "%.2f" % r["temp_mean"],
              "-" if pd.isna(r["rain_total_mm"]) else "%.1f" % r["rain_total_mm"], r["n_hari_valid"], r["n_hari_hujan"]))
md.append("\nHujan total bulanan dijumlahkan hanya dari hari valid sehingga bulan dengan hari valid sedikit bisa terlihat lebih kering dari kenyataan.\n")
md.append("## 4. Perbandingan musim (hari valid)\n")
md.append("Definisi musim: hujan = bulan %s; kemarau = bulan %s; sumber: %s; bulan lain = peralihan.\n" % (MUSIM_HUJAN, MUSIM_KEMARAU, SUMBER_MUSIM))
md.append("| Musim | n hari | Suhu harian: rata-rata / median (C) | Hujan harian: rata-rata / median / P90 (mm/hari) | Hari hujan |")
md.append("|---|---|---|---|---|")
for t in tabel_musim:
    md.append("| %s | %d | %.2f / %.2f | %.2f / %.2f / %.2f | %d |" % t)
md.append("")
md.append("## 5. Hari hujan ekstrem (>= %.0f mm/hari)\n" % HUJAN_EKSTREM_MM_HARI)
if len(ekstrem) == 0:
    md.append("Tidak ada hari valid yang mencapai ambang.\n")
else:
    for _, r in ekstrem.head(10).iterrows():
        md.append("- %s: %.1f mm/hari" % (r["tanggal"].date(), r["rain_hari"]))
    md.append("\nJumlah hari ekstrem: %d (dari %d hari valid)\n" % (len(ekstrem), len(hv)))
md.append("## 6. Interpretasi fisis\n")
md.append("- **Pemanasan siang**: suhu naik setelah matahari terbit karena radiasi surya memanaskan permukaan, lalu udara di atasnya; puncak suhu biasanya tertinggal beberapa jam dari puncak radiasi (sekitar tengah hari) karena inersia termal tanah dan udara. Puncak yang teramati pada %02d:00 konsisten dengan pola ini bila berada di siang-awal sore." % jam_puncak_suhu)
md.append("- **Konveksi**: pemanasan permukaan menaikkan ketidakstabilan atmosfer, sehingga hujan konvektif di wilayah tropis cenderung terjadi siang-sore. Jam puncak hujan teramati pada %s." % (("%02d:00" % jam_puncak_hujan) if jam_puncak_hujan is not None else "tidak dapat ditentukan"))
md.append("- **Monsun**: perbedaan jumlah hujan dan hari hujan antara musim hujan dan kemarau sejalan dengan pergeseran angin monsun (basah saat monsun yang membawa uap air, kering saat monsun dari benua yang lebih kering). Ini kesesuaian pola, bukan bukti sebab-akibat dari data ini saja.")
md.append("- Korelasi antara suhu dan hujan pada data ini **tidak** menunjukkan sebab-akibat; awan hujan juga menurunkan suhu sehingga arah hubungan dapat berlawanan.\n")
md.append("## 7. Keterbatasan\n")
md.append("- Periode data %.2f tahun (< 10 tahun): **tidak ada kesimpulan tren jangka panjang** yang dapat ditarik; satu tahun juga rentan terhadap variabilitas antartahun (mis. ENSO/IOD)." % periode_tahun)
md.append("- Data qc_ok = False dikeluarkan (%d baris) dan tidak diisi; hal ini dapat menggeser statistik bila pola datanya hilang tidak acak." % n_keluar)
md.append("- Ambang hari hujan (%.1f mm), hari valid (%.0f %%), dan ambang ekstrem (%.0f mm/hari) adalah asumsi analis." % (HARI_HUJAN_MM, MIN_FRAC_HARI * 100, HUJAN_EKSTREM_MM_HARI))
md.append("- Perbandingan dengan normal klimatologi BMKG wilayah terdekat belum dilakukan (data normal tidak tersedia); lakukan manual dan catat perbedaannya.\n")
md.append("## 8. Verifikasi\n")
for j, med, acuan, ok in hasil_cek:
    md.append("- Median suhu jam %02d:00 dihitung ulang dari CSV mentah: %.2f C; di diurnal_stats.csv: %.2f C -> **%s**." % (j, med, acuan, "SAMA" if ok else "BEDA (laporkan)"))
f = open(os.path.join(OUT_DIR, "analysis.md"), "w", encoding="utf-8"); f.write("\n".join(md)); f.close()

# ----------------------------- CEK SHA-256 SESUDAH ---------------------------
f = open(PATH_DATA, "rb"); sha_sesudah = hashlib.sha256(f.read()).hexdigest(); f.close()
if sha_sebelum != sha_sesudah:
    sys.exit("GAGAL: aws_clean.csv berubah selama proses!")

# ----------------------------- RUN CONFIG ------------------------------------
cfg = {"zona_waktu_data": ZONA_DATA, "zona_waktu_analisis": ZONA_LOKAL,
       "musim_hujan_bulan": MUSIM_HUJAN, "musim_kemarau_bulan": MUSIM_KEMARAU, "sumber_musim": SUMBER_MUSIM,
       "ambang_hujan_ekstrem_mm_per_hari": HUJAN_EKSTREM_MM_HARI, "ambang_hari_hujan_mm": HARI_HUJAN_MM,
       "min_fraksi_data_hari_valid": MIN_FRAC_HARI, "min_data_10menit_per_jam_hujan": MIN_DATA_JAM_HUJAN,
       "n_baris_input": n_total, "n_baris_dikeluarkan_qc": n_keluar,
       "periode_tahun": round(periode_tahun, 3),
       "definisi": {"rain_mean_mm": "rata-rata hujan per jam (mm/jam) dari jam dengan >= 6 data 10 menit valid",
                    "n": "jumlah observasi suhu valid pada jam tersebut", "n_jam_hujan": "jumlah pasangan (tanggal, jam) untuk rata-rata hujan"},
       "sha256_aws_clean": sha_sesudah, "sha256_identik": sha_sebelum == sha_sesudah,
       "versi": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__}}
f = open(os.path.join(OUT_DIR, "run_config.json"), "w"); json.dump(cfg, f, indent=2); f.close()

# ----------------------------- HASIL AKHIR -----------------------------------
print("Analisis AWS selesai: statistik diurnal dan bulanan disimpan di", os.path.abspath(OUT_DIR))

Analisis AWS selesai: statistik diurnal dan bulanan disimpan di c:\Users\X270\Downloads\results\aws_diurnal


 ## **Prompt 5 — Geolistrik + AWS: Curah Hujan dan ERT Time-Lapse untuk Infiltrasi**

### **[R] Role**
Bertindaklah sebagai reviewer analisis terpadu geofisika-meteorologi. Fokus pada penyelarasan waktu, definisi akumulasi hujan, dan kehati-hatian dalam klaim kausal.

### **[T] Task**
Hitung akumulasi curah hujan AWS sebelum setiap tanggal survei, hitung perubahan resistivitas relatif terhadap survei baseline pada lintasan ERT yang sama, lalu jelaskan apakah pola perubahan konsisten dengan infiltrasi hujan. Jangan menyimpulkan hubungan sebab-akibat.

### **[C] Context**
ERT diukur pada lintasan yang sama di `{{lokasi}}` sebanyak `{{3–4}}` kali (mis. akhir kemarau, awal hujan, puncak hujan). AWS terdekat berjarak sekitar `{{x}}` km. Hasil dipakai untuk studi resapan dan kerentanan lereng pada tahap eksplorasi. Hasil bukan model infiltrasi dan bukan penilaian stabilitas lereng.

### **[I] Input**
- `data/processed/aws_daily.csv`: `date`, `rain_mm` (mm/hari), `temp_c` (°C), zona waktu `{{WIB/UTC}}`.
- `data/processed/ert_t1.csv`, `ert_t2.csv`, `ert_t3.csv`: `x_m`, `z_m` (m), `rho_ohm_m` (Ω·m), grid inversi identik.
- `data/raw/survey_log.csv`: `survey_id`, `date`, `kondisi_tanah`, `rms_pct`.
- Jendela akumulasi: `{{3, 7, 14}}` hari. Survei baseline: `{{survey_id}}`.

### **[K] Constraints**
- Jangan mengubah data input; jangan menjalankan ulang inversi.
- Jika grid antar survei tidak identik, jangan resampling diam-diam: berhenti dan tanyakan.
- Perubahan resistivitas dinyatakan sebagai persen relatif terhadap baseline: `(rho_t − rho_0) / rho_0 × 100`.
- Jangan mengklaim kausalitas; bahas faktor lain (suhu tanah, litologi, kualitas inversi, jarak AWS).
- Software: Python dengan pandas dan NumPy; tanpa jaringan.

### **[O] Output (rencana teks)**
- `results/timelapse/rain_accum.csv`: `survey_id`, `date`, `accum_3d_mm`, `accum_7d_mm`, `accum_14d_mm`, `n_missing_days`.
- `results/timelapse/rho_change.csv`: `survey_id`, `x_m`, `z_m`, `rho_ohm_m`, `delta_pct`.
- `results/timelapse/analysis.md`: perubahan resistivitas per kedalaman dan zona, hubungan dengan akumulasi hujan, zona yang tidak merespons beserta dugaan penyebab, keterbatasan, rekomendasi pemantauan lanjutan. Peta perubahan dan grafik hujan vs resistivitas hanya direncanakan, belum digambar.
- `results/timelapse/run_config.json`: jendela akumulasi, baseline, zona waktu.

### **[Q] Quality criteria**
- Setiap kesimpulan menyebut data pendukung (akumulasi hujan dan persentase perubahan).
- Hari hujan hilang dilaporkan dan tidak diisi.
- Alternatif penjelasan dibahas untuk setiap zona yang berubah.
- Pilihan jendela akumulasi dinyatakan dan diberi alasan.
- Struktur laporan konsisten dari data ke interpretasi ke kesimpulan.

### **[V] Optional verification**
- Check: hitung ulang `accum_7d_mm` satu survei secara manual. Expected: sama dengan `rain_accum.csv`. Jika gagal: laporkan kesalahan penjumlahan.
- Check: ulangi analisis dengan jendela 3 dan 14 hari. Expected: arah hubungan tidak berbalik. Jika berbalik: laporkan sebagai hasil sensitif terhadap jendela.
- Check: bandingkan dengan data kelembapan tanah atau muka air sumur `{{bila tersedia}}`. Expected: arah perubahan sejalan. Jika tidak: catat sebagai ketidaksesuaian.

In [11]:
import os, sys, json, hashlib, platform
import numpy as np
import pandas as pd


# ----------------------------- PARAMETER (isi sesuai prompt) -----------------
LOKASI        = "Makassar, Sulawesi Selatan (data simulasi)"
ZONA_WAKTU    = "WITA"                  # zona waktu tanggal AWS & survei: WIB/WITA/WIT/UTC (WAJIB)
JARAK_AWS_KM  = "2.5"                 # jarak AWS ke lintasan (km)
JENDELA       = [3, 7, 14]              # jendela akumulasi (hari)
JENDELA_UTAMA = 7                       # jendela utama untuk pembahasan
BASELINE      = "T1"                    # survey_id baseline
DEPTH_BINS    = [0.0, 3.0, 6.0, 10.0, 1e9]   # batas kedalaman (m) untuk ringkasan per kedalaman
RESP_THR_PCT  = 5.0                     # |rata-rata delta| < 5 % dianggap tidak merespons (asumsi)
RMS_BATAS     = 10.0                    # RMS inversi > 10 % ditandai kualitas rendah (asumsi)
BUAT_DATA_SINTETIS = True               # True: bila file input belum ada, dibuat data SIMULASI

PATH_AWS = "data/processed/aws_daily.csv"
PATH_LOG = "data/raw/survey_log.csv"
PATH_ERT = "data/processed/ert_%s.csv"  # diisi dengan survey_id huruf kecil, mis. ert_t1.csv
OUT_DIR  = "results/timelapse"

# ----------------------------- DATA SINTETIS (bila file belum ada) -----------
# PERHATIAN: ini data simulasi untuk uji skrip, BUKAN data lapangan.
if (not os.path.exists(PATH_LOG)) and BUAT_DATA_SINTETIS:
    os.makedirs(os.path.dirname(PATH_AWS), exist_ok=True)
    os.makedirs(os.path.dirname(PATH_LOG), exist_ok=True)
    rng_s = np.random.default_rng(5)
    tgl_s = pd.date_range("2025-08-15", "2026-02-15", freq="D")
    p_s = np.where(tgl_s < pd.Timestamp("2025-11-01"), 0.08, 0.6)
    rain_s = np.where(rng_s.random(len(tgl_s)) < p_s, rng_s.exponential(np.where(tgl_s < pd.Timestamp("2025-11-01"), 4.0, 14.0)), 0.0)
    aws_s = pd.DataFrame({"date": tgl_s.strftime("%Y-%m-%d"), "rain_mm": np.round(rain_s, 1),
                          "temp_c": np.round(27 + rng_s.normal(0, 0.6, len(tgl_s)), 1)})
    aws_s.loc[aws_s["date"].isin(["2026-01-18", "2026-01-21"]), "rain_mm"] = np.nan    # hari hilang
    aws_s.to_csv(PATH_AWS, index=False)
    pd.DataFrame({"survey_id": ["T1", "T2", "T3"], "date": ["2025-10-10", "2025-11-20", "2026-01-25"],
                  "kondisi_tanah": ["kering", "lembap", "jenuh"], "rms_pct": [3.8, 4.2, 4.0]}).to_csv(PATH_LOG, index=False)
    xs_s = np.arange(0, 60.1, 3.0); ds_s = np.arange(0, 18.1, 2.0)
    faktor_s = {"t1": (0.0, 0.0), "t2": (-0.25, -0.10), "t3": (-0.45, -0.20)}
    for sid_s, (f_dangkal, f_tengah) in faktor_s.items():
        baris_s = []
        for x_s in xs_s:
            for d_s in ds_s:
                if d_s < 4: rho_s = 300.0
                elif d_s < 10: rho_s = 120.0
                else: rho_s = 600.0
                if x_s < 45 and d_s < 6: rho_s *= 1 + f_dangkal
                elif x_s < 45 and d_s < 10: rho_s *= 1 + f_tengah
                baris_s.append([x_s, 100 - 0.1 * x_s - d_s, rho_s * np.exp(rng_s.normal(0, 0.02))])
        pd.DataFrame(baris_s, columns=["x_m", "z_m", "rho_ohm_m"]).to_csv(PATH_ERT % sid_s, index=False)

for p_ in [PATH_AWS, PATH_LOG]:
    if not os.path.exists(p_):
        sys.exit("File tidak ditemukan: %s\nFolder kerja saat ini: %s" % (p_, os.getcwd()))
if ZONA_WAKTU not in ["WIB", "WITA", "WIT", "UTC"]:
    sys.exit("BERHENTI: zona waktu tidak jelas. Isi ZONA_WAKTU dengan WIB/WITA/WIT/UTC.")

# ----------------------------- BACA LOG SURVEI -------------------------------
sv = pd.read_csv(PATH_LOG)
for k in ["survey_id", "date", "kondisi_tanah", "rms_pct"]:
    if k not in sv.columns:
        sys.exit("Kolom wajib tidak ada di survey_log.csv: " + k)
sv["survey_id"] = sv["survey_id"].astype(str)
sv["date"] = pd.to_datetime(sv["date"])
sv = sv.sort_values("date").reset_index(drop=True)
if BASELINE not in sv["survey_id"].values:
    sys.exit("BERHENTI: baseline '%s' tidak ada di survey_log.csv (ada: %s)." % (BASELINE, ", ".join(sv["survey_id"])))
for sid in sv["survey_id"]:
    if not os.path.exists(PATH_ERT % sid.lower()):
        sys.exit("File ERT tidak ditemukan: %s" % (PATH_ERT % sid.lower()))

# ----------------------------- CEK SHA-256 SEBELUM ---------------------------
daftar_input = [PATH_AWS, PATH_LOG] + [PATH_ERT % sid.lower() for sid in sv["survey_id"]]
sha_sebelum = {}
for p_ in daftar_input:
    f = open(p_, "rb"); sha_sebelum[p_] = hashlib.sha256(f.read()).hexdigest(); f.close()
os.makedirs(OUT_DIR, exist_ok=True)

# ----------------------------- BACA AWS HARIAN -------------------------------
aws = pd.read_csv(PATH_AWS)
for k in ["date", "rain_mm", "temp_c"]:
    if k not in aws.columns:
        sys.exit("Kolom wajib tidak ada di aws_daily.csv: " + k)
aws["date"] = pd.to_datetime(aws["date"])
if aws["date"].duplicated().any():
    sys.exit("BERHENTI: ada tanggal duplikat di aws_daily.csv. Mohon konfirmasi cara penanganannya.")
rain = aws.set_index("date")["rain_mm"]
suhu = aws.set_index("date")["temp_c"]

# ----------------------------- AKUMULASI HUJAN -------------------------------
# akumulasi N hari = jumlah hujan pada hari (tanggal survei - N) s/d (tanggal survei - 1); hari survei tidak ikut
baris_acc = []
miss_detail = {}
for _, s in sv.iterrows():
    sd = s["date"]
    rec = {"survey_id": s["survey_id"], "date": sd.date()}
    miss = {}
    for N in JENDELA:
        hari = pd.date_range(sd - pd.Timedelta(days=N), sd - pd.Timedelta(days=1), freq="D")
        v = rain.reindex(hari)                       # hari tidak ada di file -> NaN (tidak diisi)
        rec["accum_%dd_mm" % N] = round(float(v.sum(skipna=True)), 1) if v.notna().any() else np.nan
        miss[N] = int(v.isna().sum())
    rec["n_missing_days"] = miss[max(JENDELA)]
    rec["_temp7"] = float(suhu.reindex(pd.date_range(sd - pd.Timedelta(days=7), sd - pd.Timedelta(days=1))).mean())
    miss_detail[s["survey_id"]] = miss
    baris_acc.append(rec)
df_acc_full = pd.DataFrame(baris_acc)
kolom_acc = ["survey_id", "date"] + ["accum_%dd_mm" % N for N in JENDELA] + ["n_missing_days"]
df_acc = df_acc_full[kolom_acc]
df_acc.to_csv(os.path.join(OUT_DIR, "rain_accum.csv"), index=False)

# ----------------------------- LOAD ERT & CEK GRID ---------------------------
ert = {}
for sid in sv["survey_id"]:
    e = pd.read_csv(PATH_ERT % sid.lower())
    for k in ["x_m", "z_m", "rho_ohm_m"]:
        if k not in e.columns:
            sys.exit("Kolom wajib tidak ada di %s: %s" % (PATH_ERT % sid.lower(), k))
    ert[sid] = e.sort_values(["x_m", "z_m"]).reset_index(drop=True)
ref = ert[BASELINE]
for sid in ert:
    e = ert[sid]
    if len(e) != len(ref) or (not np.allclose(e["x_m"].values, ref["x_m"].values)) or (not np.allclose(e["z_m"].values, ref["z_m"].values)):
        sys.exit("BERHENTI: grid inversi survei %s tidak identik dengan baseline %s. Tidak dilakukan resampling; mohon konfirmasi." % (sid, BASELINE))

# ----------------------------- PERUBAHAN RESISTIVITAS ------------------------
rho0 = ref["rho_ohm_m"].values.astype(float)
depth = (ref.groupby("x_m")["z_m"].transform("max") - ref["z_m"]).values
kel_dalam = np.digitize(depth, DEPTH_BINS[1:-1])        # indeks bin kedalaman
q1, q2 = np.nanpercentile(rho0, [33.3, 66.7])
zona_idx = np.where(rho0 < q1, 0, np.where(rho0 < q2, 1, 2))
nama_zona = ["rho rendah (baseline)", "rho menengah (baseline)", "rho tinggi (baseline)"]

df_chg_list = []
delta_mat = {}
for sid in sv["survey_id"]:
    r = ert[sid]["rho_ohm_m"].values.astype(float)
    delta = np.where(rho0 > 0, (r - rho0) / rho0 * 100.0, np.nan)
    delta_mat[sid] = delta
    df_chg_list.append(pd.DataFrame({"survey_id": sid, "x_m": ref["x_m"].values, "z_m": ref["z_m"].values,
                                     "rho_ohm_m": np.round(r, 3), "delta_pct": np.round(delta, 3)}))
df_chg = pd.concat(df_chg_list, ignore_index=True)
df_chg.to_csv(os.path.join(OUT_DIR, "rho_change.csv"), index=False)

# ----------------------------- RINGKASAN ANALISIS ----------------------------
sid_lain = [s for s in sv["survey_id"] if s != BASELINE]
label_bin = []
for b in range(len(DEPTH_BINS) - 1):
    label_bin.append("%.0f - %s m" % (DEPTH_BINS[b], ("%.0f" % DEPTH_BINS[b + 1]) if DEPTH_BINS[b + 1] < 1e8 else "dst"))

mean_depth = {}      # [sid][bin] -> rata-rata delta
for sid in sid_lain:
    mean_depth[sid] = [float(np.nanmean(delta_mat[sid][kel_dalam == b])) if np.any(kel_dalam == b) else np.nan for b in range(len(label_bin))]
mean_zona = {}
frac_resp = {}
for sid in sid_lain:
    mean_zona[sid] = [float(np.nanmean(delta_mat[sid][zona_idx == z])) for z in range(3)]
    frac_resp[sid] = float(np.mean(np.abs(delta_mat[sid][~np.isnan(delta_mat[sid])]) >= RESP_THR_PCT) * 100.0)

# zona (kedalaman x posisi) yang tidak merespons pada semua survei non-baseline
x_u = np.sort(ref["x_m"].unique())
x_bagi = np.array_split(x_u, 3)
tdk_resp = []
for b in range(len(label_bin)):
    for k_, xs_ in enumerate(x_bagi):
        m = (kel_dalam == b) & np.isin(ref["x_m"].values, xs_)
        if m.sum() == 0:
            continue
        nilai = [float(np.nanmean(delta_mat[sid][m])) for sid in sid_lain]
        if all([abs(v) < RESP_THR_PCT for v in nilai]):
            tdk_resp.append((label_bin[b], xs_.min(), xs_.max(), nilai, float(np.nanmean(rho0[m]))))

# ----------------------------- KONSISTENSI DENGAN INFILTRASI -----------------
acc_u = df_acc_full.set_index("survey_id")["accum_%dd_mm" % JENDELA_UTAMA]
konsisten = {}
for sid in sid_lain:
    dlt = acc_u[sid] - acc_u[BASELINE]
    obs = mean_depth[sid][0]                  # bin dangkal
    if np.isnan(dlt) or np.isnan(obs):
        konsisten[sid] = (dlt, obs, "tidak dapat dinilai (data hujan/ERT tidak lengkap)")
    elif dlt == 0 or abs(obs) < RESP_THR_PCT:
        konsisten[sid] = (dlt, obs, "tidak ada respons berarti pada bin dangkal")
    else:
        harapan = -np.sign(dlt)               # hujan lebih banyak dari baseline -> resistivitas turun
        konsisten[sid] = (dlt, obs, "konsisten dengan infiltrasi" if np.sign(obs) == harapan else "TIDAK konsisten dengan infiltrasi")

# sensitivitas jendela: kemiringan hubungan (akumulasi vs rata-rata delta dangkal), semua survei termasuk baseline (delta 0)
sens = []
for N in JENDELA:
    xs_ = []; ys_ = []
    for sid in sv["survey_id"]:
        a = df_acc_full.set_index("survey_id").loc[sid, "accum_%dd_mm" % N]
        y = 0.0 if sid == BASELINE else mean_depth[sid][0]
        if not (np.isnan(a) or np.isnan(y)):
            xs_.append(a); ys_.append(y)
    if len(xs_) >= 3 and np.ptp(xs_) > 0:
        sens.append((N, float(np.polyfit(xs_, ys_, 1)[0]), len(xs_)))
    else:
        sens.append((N, np.nan, len(xs_)))
tanda = [np.sign(s[1]) for s in sens if not np.isnan(s[1])]
arah_stabil = (len(tanda) > 0) and (len(set(tanda)) == 1)

# ----------------------------- VERIFIKASI: hitung ulang akumulasi 7 hari -----
sid_cek = sid_lain[0] if len(sid_lain) > 0 else BASELINE
tgl_cek = sv.loc[sv["survey_id"] == sid_cek, "date"].iloc[0]
manual = 0.0
for k_ in range(1, 8):
    tg = tgl_cek - pd.Timedelta(days=k_)
    if tg in rain.index and not pd.isna(rain.loc[tg]):
        manual += float(rain.loc[tg])
acuan = float(df_acc_full.set_index("survey_id").loc[sid_cek, "accum_7d_mm"]) if 7 in JENDELA else np.nan
cek_ok = abs(round(manual, 1) - acuan) < 1e-9 if not np.isnan(acuan) else None

# ----------------------------- ANALISIS (Markdown) ---------------------------
md = []
md.append("# Analisis Time-Lapse ERT dan Curah Hujan AWS\n")
md.append("Lokasi: %s | Zona waktu: %s | Jarak AWS ke lintasan: %s km | Baseline: **%s**\n" % (LOKASI, ZONA_WAKTU, JARAK_AWS_KM, BASELINE))
md.append("> Eksplorasi awal. Bukan model infiltrasi dan bukan penilaian stabilitas lereng. Tidak ada klaim sebab-akibat.\n")
md.append("## 1. Data dan kualitas survei\n")
md.append("| Survei | Tanggal | Kondisi tanah | RMS inversi (%) | Catatan |")
md.append("|---|---|---|---|---|")
for _, s in sv.iterrows():
    md.append("| %s | %s | %s | %.2f | %s |" % (s["survey_id"], s["date"].date(), s["kondisi_tanah"], s["rms_pct"],
              "RMS tinggi, hati-hati" if s["rms_pct"] > RMS_BATAS else "-"))
md.append("")
md.append("- Jumlah sel per survei: %d (grid identik antar survei; tidak ada resampling)." % len(ref))
md.append("- Perubahan = (rho_t - rho_0) / rho_0 x 100 %%, rho_0 dari baseline %s.\n" % BASELINE)
md.append("## 2. Akumulasi curah hujan sebelum survei\n")
md.append("Akumulasi N hari = jumlah hujan pada N hari sebelum tanggal survei (hari survei tidak ikut). Hari hilang **tidak diisi**; akumulasi adalah jumlah hari yang tersedia (batas bawah).\n")
md.append("| Survei | Tanggal | " + " | ".join(["%d hari (mm)" % N for N in JENDELA]) + " | Hari hilang per jendela | Suhu rata-rata 7 hari (C) |")
md.append("|---|---|" + "---|" * len(JENDELA) + "---|---|")
for _, r in df_acc_full.iterrows():
    md.append("| %s | %s | " % (r["survey_id"], r["date"]) + " | ".join(["%.1f" % r["accum_%dd_mm" % N] if not pd.isna(r["accum_%dd_mm" % N]) else "-" for N in JENDELA])
              + " | " + ", ".join(["%dd: %d" % (N, miss_detail[r["survey_id"]][N]) for N in JENDELA]) + " | %.1f |" % r["_temp7"])
md.append("")
md.append("Alasan jendela: 3 hari mewakili respons cepat zona dangkal; 7 hari (jendela utama) mewakili kebasahan tanah mingguan; 14 hari mewakili kondisi antesedan yang lebih lama dan zona lebih dalam. Pemilihan ini asumsi analis dan diuji sensitivitasnya (bagian 6).\n")
md.append("## 3. Perubahan resistivitas per kedalaman (rata-rata delta %, relatif terhadap baseline)\n")
md.append("| Survei | " + " | ".join(label_bin) + " | Sel berubah >= %.0f %% (%%) |" % RESP_THR_PCT)
md.append("|---|" + "---|" * len(label_bin) + "---|")
for sid in sid_lain:
    md.append("| %s | " % sid + " | ".join(["%.1f" % v if not np.isnan(v) else "-" for v in mean_depth[sid]]) + " | %.1f |" % frac_resp[sid])
md.append("")
md.append("## 4. Perubahan per zona (kelas resistivitas baseline)\n")
md.append("Batas zona dari persentil baseline: rendah < %.0f; menengah %.0f - %.0f; tinggi >= %.0f ohm.m.\n" % (q1, q1, q2, q2))
md.append("| Survei | " + " | ".join(nama_zona) + " |")
md.append("|---|---|---|---|")
for sid in sid_lain:
    md.append("| %s | " % sid + " | ".join(["%.1f %%" % v for v in mean_zona[sid]]) + " |")
md.append("")
md.append("## 5. Hubungan dengan akumulasi hujan (jendela utama %d hari)\n" % JENDELA_UTAMA)
for sid in sid_lain:
    dlt, obs, teks = konsisten[sid]
    md.append("- %s vs baseline: selisih akumulasi %s mm; rata-rata perubahan resistivitas lapisan dangkal (%s) %s %% -> %s." % (
        sid, "%.1f" % dlt if not np.isnan(dlt) else "-", label_bin[0], "%.1f" % obs if not np.isnan(obs) else "-", teks))
md.append("")
md.append("Harapan bila ada infiltrasi: akumulasi lebih besar dari baseline -> resistivitas dangkal turun; lebih kecil -> naik. Kesesuaian pola **tidak membuktikan** infiltrasi sebagai penyebab.\n")
md.append("## 6. Sensitivitas jendela akumulasi\n")
md.append("| Jendela (hari) | Kemiringan (% perubahan per mm hujan) | n survei |")
md.append("|---|---|---|")
for N, sl, n in sens:
    md.append("| %d | %s | %d |" % (N, "%.4f" % sl if not np.isnan(sl) else "tidak dapat dihitung", n))
if len(tanda) == 0:
    md.append("\nArah hubungan tidak dapat dinilai (data tidak cukup).\n")
elif arah_stabil:
    md.append("\nArah hubungan **tidak berbalik** antar jendela (stabil). Dengan hanya %d survei, ini indikasi lemah, bukan uji statistik.\n" % len(sv))
else:
    md.append("\n**Arah hubungan berbalik antar jendela: hasil sensitif terhadap pilihan jendela.**\n")
md.append("## 7. Zona yang tidak merespons\n")
if len(tdk_resp) == 0:
    md.append("Tidak ada zona (kedalaman x sepertiga lintasan) dengan |perubahan| < %.0f %% pada semua survei.\n" % RESP_THR_PCT)
else:
    md.append("Zona dengan |rata-rata perubahan| < %.0f %% pada semua survei non-baseline:\n" % RESP_THR_PCT)
    for lb, xa, xb, nil, rr in tdk_resp:
        md.append("- Kedalaman %s, x = %.0f - %.0f m (rho baseline rata-rata %.0f ohm.m; perubahan: %s)." % (
            lb, xa, xb, rr, ", ".join(["%s %.1f %%" % (sid_lain[i], nil[i]) for i in range(len(nil))])))
    md.append("\nDugaan penyebab (belum diuji): (a) material kedap atau batuan kompak sehingga air tidak meresap; (b) zona lebih dalam dari jangkauan perubahan musiman; (c) sensitivitas inversi rendah/di luar DOI; (d) air sudah jenuh sejak baseline; (e) hujan lokal berbeda dengan AWS (jarak %s km).\n" % JARAK_AWS_KM)
md.append("## 8. Alternatif penjelasan untuk zona yang berubah\n")
md.append("- **Suhu tanah**: resistivitas turun sekitar 2 % per kenaikan 1 C; suhu rata-rata 7 hari sebelum survei tercantum di bagian 2 (selisih suhu antar survei dapat ikut menggeser nilai).")
md.append("- **Litologi/porositas**: tanah berpori lebih responsif terhadap kadar air daripada batuan kompak; perbedaan respons antar zona dapat mencerminkan litologi, bukan hujan.")
md.append("- **Kualitas inversi**: perbedaan RMS dan regularisasi antar survei dapat menimbulkan perubahan semu; survei dengan RMS tinggi harus dibaca hati-hati.")
md.append("- **Jarak AWS (%s km)**: hujan di lintasan dapat berbeda dengan hujan di AWS." % JARAK_AWS_KM)
md.append("- **Kualitas air dan kejadian lain**: salinitas air, muka air tanah, vegetasi, atau pemadatan/pengolahan lahan juga mengubah resistivitas.\n")
md.append("## 9. Keterbatasan\n")
md.append("- Hanya %d survei; tidak cukup untuk menarik hubungan statistik, apalagi sebab-akibat." % len(sv))
md.append("- Data kelembapan tanah/muka air sumur tidak dipakai (belum tersedia): bandingkan bila ada.")
md.append("- Grafik (peta perubahan dan hujan vs resistivitas) hanya direncanakan, belum dibuat.\n")
md.append("## 10. Rekomendasi pemantauan lanjutan\n")
md.append("- Tambah survei pada kondisi berbeda dan interval lebih rapat setelah hujan besar.")
md.append("- Pasang sensor kelembapan tanah/muka air dan pluviometer dekat lintasan.")
md.append("- Pakai parameter inversi dan model awal identik; catat suhu tanah saat survei.\n")
md.append("## 11. Verifikasi\n")
if cek_ok is None:
    md.append("- Jendela 7 hari tidak dipakai; verifikasi manual dilewati.")
else:
    md.append("- Hitung ulang manual accum_7d_mm survei %s: %.1f mm; di rain_accum.csv: %.1f mm -> **%s**." % (sid_cek, manual, acuan, "SAMA" if cek_ok else "BEDA (laporkan kesalahan penjumlahan)"))
md.append("- Uji jendela 3 dan 14 hari: %s." % ("arah hubungan tidak berbalik" if arah_stabil else "arah berbalik atau tidak dapat dinilai (lihat bagian 6)"))
md.append("- Perbandingan dengan kelembapan tanah/muka air sumur: belum dilakukan (data tidak tersedia).")
f = open(os.path.join(OUT_DIR, "analysis.md"), "w", encoding="utf-8"); f.write("\n".join(md)); f.close()

# ----------------------------- CEK SHA-256 SESUDAH ---------------------------
for p_ in daftar_input:
    f = open(p_, "rb"); sesudah = hashlib.sha256(f.read()).hexdigest(); f.close()
    if sesudah != sha_sebelum[p_]:
        sys.exit("GAGAL: %s berubah selama proses!" % p_)

# ----------------------------- RUN CONFIG ------------------------------------
cfg = {"jendela_akumulasi_hari": JENDELA, "jendela_utama_hari": JENDELA_UTAMA, "baseline": BASELINE, "zona_waktu": ZONA_WAKTU,
       "definisi_akumulasi": "jumlah rain_mm pada N hari sebelum tanggal survei (hari survei tidak ikut); hari hilang tidak diisi",
       "alasan_jendela": {"3": "respons cepat zona dangkal", "7": "kebasahan tanah mingguan (jendela utama)", "14": "kondisi antesedan dan zona lebih dalam"},
       "rumus_perubahan": "(rho_t - rho_0) / rho_0 * 100",
       "batas_kedalaman_m": [b for b in DEPTH_BINS if b < 1e8], "batas_respons_pct": RESP_THR_PCT, "batas_rms_pct": RMS_BATAS,
       "grid_identik": True, "resampling": False, "inversi_dijalankan_ulang": False,
       "sha256_input": sha_sebelum,
       "versi": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__}}
f = open(os.path.join(OUT_DIR, "run_config.json"), "w"); json.dump(cfg, f, indent=2); f.close()

# ----------------------------- HASIL AKHIR -----------------------------------
print("Time-lapse selesai: akumulasi hujan dan perubahan resistivitas disimpan di", os.path.abspath(OUT_DIR))

Time-lapse selesai: akumulasi hujan dan perubahan resistivitas disimpan di c:\Users\X270\Downloads\results\timelapse
